>## <연구 목표>
>### 목적
>모델의 예측 신호 자체의 우수성이 아니라, “해당 시점의 예측을 믿어도 되는가”를 정량화하는 신뢰도 지표(C-index)를 설계하고 정당성을 입증.
>### 요구사항
특정 종목/특정 모델에 종속되지 않도록, 여러 자산군/여러 모델(혹은 최소 2개 계열)에서 동일한 결론이 유지되어야 함 (겨울 방학 동안에는 여러 자산군에 대한 실험을 우선적으로 마무리하며, 모델은 LightGBM으로 고정할 예정)

---


In [ ]:
from google.colab import drive

DRIVE_AVAILABLE = False
try:
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
except Exception as exc:
    print('Google Drive mount unavailable; using /content/c-index for this run.')
    print(type(exc).__name__, str(exc))


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path('/content/drive/MyDrive/c-index') if DRIVE_AVAILABLE else Path('/content/c-index')
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
RESULTS_DIR = PROJECT_DIR / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
TABLES_DIR = RESULTS_DIR / 'tables'

for path in [ARTIFACT_DIR, FIGURES_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('ARTIFACT_DIR:', ARTIFACT_DIR)
print('FIGURES_DIR:', FIGURES_DIR)
print('TABLES_DIR:', TABLES_DIR)


In [ ]:
# Install only packages missing from the current Colab runtime.
import importlib.util
import importlib.metadata
import json
import subprocess
import sys

required_packages = ['yfinance', 'lightgbm', 'shap']
missing_packages = [p for p in required_packages if importlib.util.find_spec(p) is None]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])

# Record the exact environment used by this run.
tracked_packages = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'lightgbm', 'torch', 'shap', 'yfinance', 'matplotlib', 'joblib']
environment_versions = {name: importlib.metadata.version(name) for name in tracked_packages}
with open(ARTIFACT_DIR / 'environment_versions.json', 'w', encoding='utf-8') as f:
    json.dump(environment_versions, f, ensure_ascii=False, indent=2, sort_keys=True)

# Fresh training run: old model weights are intentionally not preloaded.
print('Fresh training mode. Installed missing packages:', missing_packages)
print('Environment versions:', environment_versions)


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

GLOBAL_SEED = 42

def set_global_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(GLOBAL_SEED)
print('Global seed fixed:', GLOBAL_SEED)


## <mark> 1) 데이터 구성</mark>


### 1-1) 자산 구성 (종목 특정성 제거)

* 단일 종목이 아니라, 서로 다른 성격의 자산을 패널로 구성
  * 예: 대표 지수(주식), 섹터 ETF, 채권 ETF, 원자재 ETF 등
  * 목표: 특정 자산에서는 잘 되는데, 다른 자산에서는 잘 안되는 자산에 따른 종속성을 줄이고, C-index가 공통적으로 작동함을 보이기

### 1-2) 데이터 셋 구성
> * **데이터 소스**: yfinance (daily frequency, adjusted prices)
> * **기간**: 2012-01-01 ~ 2025-12-31 (약 14년)

#### $\color{red}{\text{<Input features>}}$
#### a) Return-based features

* **1일 / 5일 / 10일 log return**

  * 가장 기본적인 가격 변화 요약 지표로, 자산·모델에 대한 의존성이 낮음
  * 단기(1d)와 중기(5d, 10d) 수익률을 함께 사용함으로써
    서로 다른 투자 시계열 스케일에서의 신호를 동시에 반영
  * 설명 관점에서 “모델이 단순 가격 변화에 의존했는가”를 점검하는 기준 피처 역할


#### b) Volatility features

* **Rolling standard deviation (20일, 60일)**

  * 단기/중기 변동성 수준을 정량화하여 시장 불확실성 상태를 반영
  * 동일한 수익률 신호라도 변동성 국면에 따라 해석이 달라질 수 있으므로,
    설명 기법 간 중요도 차이가 발생하기 쉬운 피처

* **ATR(14)**

  * 고가–저가 범위를 포함한 실질적 가격 변동 폭 지표
  * 단순 수익률 기반 변동성과 다른 관점을 제공하여
    XAI 랭킹 불일치 가능성을 의도적으로 포함


#### c) Trend features

* **EMA(12), EMA(26)**

  * 가격의 방향성과 추세 지속성을 요약하는 대표적 지표
  * 단기/중기 EMA를 함께 사용하여 추세 강도의 상대 비교 가능

* **MACD (Moving Average Convergence Divergence)**

  * 두 EMA 간 차이를 통해 추세 전환 및 가속 여부를 포착: EMA12 - EMA26
  * 모멘텀/추세 경계 영역에 위치한 피처로,
    설명기법 간 해석 차이가 자주 발생하는 후보


#### d)  Momentum features

* **RSI(14)**

* **Rate of Change (ROC)**

  * 가격 변화 속도 자체를 측정
  * 동일한 방향의 가격 움직임이라도 가속/둔화 국면을 구분할 수 있어
    feature importance 랭킹 변동성 유도에 적합

#### e) Volume features

* **Volume rolling z-score (20일 기준)**
  * 최근 20일 동안의 ‘평소 거래량’과 비교해서 오늘 거래량이 얼마나 비정상적으로 많은지를 표준화한 값

#### f) Market regime proxies (외생 변수)

* **VIX index (^VIX)**

  * S&P 500 지수 옵션 가격을 이용해 계산된 향후 약 30일간의 시장 변동성 기대치
  * 개별 자산 가격과 독립적인 외생 레짐 정보 제공

* **Dollar proxy: UUP**

  * 글로벌 달러 강세/약세 환경을 반영
  * 주식·원자재 등 다양한 자산군에 공통적으로 영향을 미치는 거시 요인

* **Bond ETFs: IEF, TLT**

  * 금리 및 안전자산 선호를 간접적으로 반영
  * 위험자산 대비 자금 이동(regime shift)을 포착하기 위한 보조 변수

#### $\color{red}{\text{<Output Label>}}$

#### a) 목적
다음 시점(T+1)의 방향성 예측을 위한 이진 분류

#### b) 라벨 정의
* $$y_t = 1 \quad \text{if } r_{t+1} > \tau$$  
* $$y_t = 0 \quad \text{if } r_{t+1} < -\tau$$  
* $$\lvert r_{t+1} \rvert \le \tau \quad \Rightarrow \quad \textbf{no-trade (excluded)}$$

* 임계값 $\tau$ (자산별 변동성 스케일을 반영한 동적 임계값)  
  $$\tau = k \times \sigma_{t}^{(20)}, \qquad \sigma_{t}^{(20)} = \text{rolling volatility over 20 days}$$

#### c) 라벨 정의 근거
* 단순 방향성 라벨은 금융 시계열의 잡음을 과도하게 포함할 위험이 있음. 변동성 기반 임계값을 사용한 이벤트 라벨링은 명확한 가격 움직임이 발생한 시점만 학습 대상으로 제한하기 때문에, 설명 기법 간 랭킹 불일치가 단순한 시장 잡음이 아니라 해당 시점 예측 결정 근거의 불안정성을 반영하는 신호인지 여부를 검증할 수 있도록 함.


In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np
import yfinance as yf

# 0) Configuration
assets = ['SPY', 'TLT', 'GLD']
regimeTickers = ['^VIX', 'IEF', 'UUP']
startDate = '2012-01-01'
endDate = '2025-12-31'

# Reproducibility guard for data source:
# yfinance can revise adjusted OHLCV retrospectively. For strict reproducibility,
# freeze downloaded raw data as CSV and reuse it unless REFRESH_DATA_CACHE=True.
REFRESH_DATA_CACHE = False

DATA_CACHE_DIR = PROJECT_DIR / 'data_cache'

DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

needed = ['Open', 'High', 'Low', 'Close', 'Volume']

def _safe_ticker_name(ticker):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', ticker)

def _cache_path(ticker):
    name = _safe_ticker_name(ticker)
    return DATA_CACHE_DIR / f'{name}_{startDate}_{endDate}_auto_adjust_true.csv'

def _normalize_ohlcv(df, ticker):
    if df is None or df.empty:
        return pd.DataFrame(columns=needed)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{ticker}: missing columns {missing}")

    df = df[needed].copy()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='last')]
    df[needed] = df[needed].apply(pd.to_numeric, errors='coerce')
    return df

# 1) Data Download with deterministic local/Drive cache
def downloadOhlcv(ticker):
    cache_file = _cache_path(ticker)

    if cache_file.exists() and not REFRESH_DATA_CACHE:
        df = pd.read_csv(cache_file, index_col=0, parse_dates=True)
        df = _normalize_ohlcv(df, ticker)
        print(f"[cache] {ticker}: {len(df)} rows <- {cache_file.name}")
        return df

    set_global_seed(GLOBAL_SEED)
    df = yf.download(
        ticker,
        start=startDate,
        end=endDate,
        interval='1d',
        auto_adjust=True,
        progress=False,
        threads=False
    )
    df = _normalize_ohlcv(df, ticker)

    if df.empty:
        raise ValueError(f"{ticker}: downloaded data is empty")

    df.to_csv(cache_file, float_format='%.12g')
    print(f"[download] {ticker}: {len(df)} rows -> {cache_file.name}")
    return df

tickersAll = list(assets + regimeTickers)
priceData = {t: downloadOhlcv(t) for t in tickersAll}

# 2) Feature Engineering
def computeFeatures(df):
    df = df.copy()
    close, high, low, vol = df['Close'], df['High'], df['Low'], df['Volume']

    # Returns
    df['ret_1d']  = np.log(close).diff(1)
    df['ret_5d']  = np.log(close).diff(5)
    df['ret_10d'] = np.log(close).diff(10)

    # Volatility
    df['vol_20'] = df['ret_1d'].rolling(20).std()
    df['vol_60'] = df['ret_1d'].rolling(60).std()

    prevClose = close.shift(1)
    tr = np.maximum(high - low, np.maximum((high - prevClose).abs(), (low - prevClose).abs()))
    df['atr_14'] = tr.rolling(14).mean() / close

    # Trend Indicators
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    df['ema_12'] = ema12 / ema26 - 1
    df['ema_26'] = ema26 / close - 1
    df['macd']   = (ema12 - ema26) / close

    # Momentum Indicators
    delta = close.diff()
    up, down = delta.clip(lower=0), (-delta).clip(lower=0)
    rollUp = up.ewm(span=14, adjust=False).mean()
    rollDown = down.ewm(span=14, adjust=False).mean()
    rs = rollUp / rollDown.replace(0, np.nan)
    df['rsi_14'] = 100 - (100 / (1 + rs))
    df['roc_10'] = close.pct_change(10)

    # Volume Analysis
    df['volume_z20'] = (vol - vol.rolling(20).mean()) / vol.rolling(20).std()

    return df

# 3) Regime Proxy Setup
regimeClose = {}
for proxy in regimeTickers:
    d = priceData[proxy]
    if d.empty: continue
    s = d['Close'].copy().sort_index()
    s = np.log(s).diff(1) if proxy == '^VIX' else np.log(s).diff(5)
    s.name = proxy
    regimeClose[proxy] = s

# 4) Panel Construction
frames = []
for asset in assets:
    dfa = computeFeatures(priceData[asset])
    dfa['asset'] = asset
    for proxy in regimeTickers:
        dfa[proxy] = regimeClose[proxy].reindex(dfa.index).ffill()
    frames.append(dfa)

panelDf = pd.concat(frames, axis=0).reset_index()
panelDf = panelDf.rename(columns={'Date': 'date', 'index': 'date'}, errors='ignore')

# 5) Data Cleaning
featureCols = ['ret_1d','ret_5d','ret_10d','vol_20','vol_60','atr_14','ema_12','ema_26','macd','rsi_14','roc_10','volume_z20'] + regimeTickers
panelDf = panelDf.dropna(subset=featureCols).reset_index(drop=True)

print("=== DATA CACHE DIR ===")
print(DATA_CACHE_DIR)
print("=== PANEL SUMMARY ===")
print(panelDf.groupby('asset').size())



In [ ]:
# 7) Label Generation
k = 0.65 # Dynamic threshold multiplier
panelDf = panelDf.sort_values(['asset', 'date']).reset_index(drop=True)

# Target: Next-day return
panelDf['ret_1d_next'] = panelDf.groupby('asset')['ret_1d'].shift(-1)
panelDf['tau'] = k * panelDf['vol_20']

# Binary Labeling (based on volatility threshold)
panelDf['y'] = np.nan
panelDf.loc[panelDf['ret_1d_next'] >  panelDf['tau'], 'y'] = 1
panelDf.loc[panelDf['ret_1d_next'] < -panelDf['tau'], 'y'] = 0

labeledDf = panelDf.dropna(subset = ['y']).reset_index(drop=True)

print("=== LABELED DATA SUMMARY ===")
print(labeledDf.groupby(['asset', 'y']).size())

---

## <mark> 2) 모델 설계 </mark>
### 2-1) LightGBM


In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report

df = labeledDf.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date', 'asset']).reset_index(drop=True)
df['asset_cat'] = df['asset'].astype('category')

# Feature Selection (excluding look-ahead features)
idCols = {'asset', 'date'}
leakageCols = {'y', 'ret_1d_next', 'tau'}
safeFeatures = [c for c in featureCols if c not in leakageCols and c not in idCols] + ['asset_cat']

X = df[safeFeatures]
y = df['y'].astype(int)

print(f"Samples: {len(df)} | Features: {len(safeFeatures)}")

In [ ]:
# Time-consistent split (Last 20% for validation)
uniqueDates = np.array(sorted(df['date'].unique()))
cut = int(len(uniqueDates) * 0.8)

trainMask = df['date'].isin(uniqueDates[:cut])
validMask = df['date'].isin(uniqueDates[cut:])

XTrain, yTrain = X.loc[trainMask].copy(), y.loc[trainMask].copy()
XValid, yValid = X.loc[validMask].copy(), y.loc[validMask].copy()

# Categorical category alignment
cats = df['asset_cat'].cat.categories
XTrain['asset_cat'] = XTrain['asset_cat'].cat.set_categories(cats)
XValid['asset_cat'] = XValid['asset_cat'].cat.set_categories(cats)

print(f"Train: {XTrain.index.min()} to {XTrain.index.max()}")
print(f"Valid: {XValid.index.min()} to {XValid.index.max()}")

In [ ]:
set_global_seed(GLOBAL_SEED)

trainData = lgb.Dataset(XTrain, label=yTrain, categorical_feature=['asset_cat'])
validData = lgb.Dataset(XValid, label=yValid, categorical_feature=['asset_cat'])

params = {
    'objective': 'binary', 'boosting_type': 'gbdt', 'learning_rate': 0.01,
    'num_leaves': 31, 'max_depth': 6, 'min_data_in_leaf': 20, 'metric': ['auc'],
    'lambda_l1': 0.5, 'lambda_l2': 5.0, 'verbosity': -1,
    'seed': GLOBAL_SEED,
    'feature_fraction_seed': GLOBAL_SEED,
    'bagging_seed': GLOBAL_SEED,
    'data_random_seed': GLOBAL_SEED,
    'drop_seed': GLOBAL_SEED,
    'deterministic': True,
    'force_col_wise': True,
    'num_threads': 1
}

model_gbm = lgb.train(
    params, trainData, num_boost_round=5000,
    valid_sets=[validData], callbacks=[lgb.early_stopping(300)]
)

pValid = model_gbm.predict(XValid, num_iteration=model_gbm.best_iteration)
print(f"Valid AUC: {roc_auc_score(yValid, pValid):.4f}")

In [ ]:
# 7) Threshold Tuning
# Objective: Optimize for Balanced Accuracy using quantile candidates
thrCandidates = np.quantile(pValid, np.linspace(0.05, 0.95, 181))

optThr = 0.5
optBacc = -np.inf

for thr in thrCandidates:
    yHat = (pValid >= thr).astype(int)
    bacc = balanced_accuracy_score(yValid, yHat)

    if bacc > optBacc:
        optBacc = bacc
        optThr = thr

# Binary conversion using optimal threshold
yHat05 = (pValid >= 0.5).astype(int)
yHatOptThr = (pValid >= optThr).astype(int)

print("Optimal threshold (Balanced Accuracy):", round(float(optThr), 4))
print("Balanced Accuracy:", round(optBacc, 4))
print("Predicted 1 rate:", round(float(yHatOptThr.mean()), 4))

In [ ]:
# 8) Evaluation Report Function
def report(y, yHat, pPred, name):
    auc  = roc_auc_score(y, pPred)
    acc  = accuracy_score(y, yHat)
    bacc = balanced_accuracy_score(y, yHat)
    f1   = f1_score(y, yHat)
    cm   = confusion_matrix(y, yHat)
    pos_rate = float(np.mean(yHat))

    print("\n" + "="*60)
    print(f"{name}")
    print("="*60)
    print(f"[Overall]")
    print(f"  AUC               : {auc:.4f}")
    print(f"  Accuracy          : {acc:.4f}")
    print(f"  Balanced Accuracy : {bacc:.4f}")
    print(f"  F1-score          : {f1:.4f}")
    print(f"  Predicted 1 rate  : {pos_rate:.4f}")

    print("\n[Confusion Matrix]")
    print("          Pred 0    Pred 1")
    print(f"True 0     {cm[0,0]:6d}    {cm[0,1]:6d}")
    print(f"True 1     {cm[1,0]:6d}    {cm[1,1]:6d}")

# 9) Performance Output
report(yValid, yHat05, pValid, "Validation Set (Threshold: 0.5)")
report(yValid, yHatOptThr, pValid, f"Validation Set (Optimal Threshold: {optThr:.4f})")

print("\n[Classification Report at Optimal Threshold]")
print(classification_report(yValid, yHatOptThr, digits=4))

Balanced Accuracy 기준으로 임계값을 튜닝한 결과, 고정 임계값(0.5) 대비 클래스 불균형을 일부 보정한 의사결정 기준을 확보하였다. GBM의 경우 검증셋 AUC는 0.5089로 낮지만, optimal threshold 0.5476 적용 시 Balanced Accuracy가 0.4960에서 0.5289로 개선되며, 예측 신호를 보다 보수적으로 선택하는 구조가 형성되었다.

다만 세 모델 모두 검증셋 AUC가 GBM 0.5089, RF 0.5216, MLP 0.5364 수준에 머물러, 모델의 예측 확률 자체가 다음 시점 방향성을 강하게 구분한다고 보기는 어렵다. 따라서 본 연구의 핵심은 강한 예측 모델을 구축했다는 주장보다는, **약한 예측 모델이 산출한 신호 중 어느 시점의 판단을 상대적으로 더 신뢰할 수 있는가**를 설명 합의도 관점에서 선별하는 데 있다.

이후 분석에서는 예측 성능과 분리된 관점에서, 시장 변동성 확대 또는 레짐 변화 구간에서 모델의 판단 근거가 시점별로 불안정해질 수 있으며 이러한 불안정성이 서로 다른 설명 기법 간 중요도 랭킹 불일치로 관측된다는 점에 주목한다. 이에 따라 특정 시점의 예측 확률이 아니라, 해당 시점의 예측 판단이 구조적으로 일관되고 신뢰 가능한 상태였는지를 평가하기 위한 설명 합의 기반 신뢰도 지표(C-index)의 필요성이 제기된다.



### 2-2) RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score,
    confusion_matrix, classification_report
)

# 1) Load data
df_rf = labeledDf.copy()
df_rf['date'] = pd.to_datetime(df_rf['date'])
df_rf = df_rf.sort_values(['date', 'asset']).reset_index(drop=True)
df_rf['asset_cat'] = df_rf['asset'].astype('category')

# 2) Feature selection to prevent data leakage
idCols = {'asset', 'date'}
leakageCols = {'y', 'ret_1d_next', 'tau'}
safeFeatures_rf = [c for c in featureCols if c not in leakageCols and c not in idCols]
safeFeatures_rf = safeFeatures_rf + ['asset_cat']

X_rf = df_rf[safeFeatures_rf].copy()
y_rf = df_rf['y'].astype(int)

# 3) Time-consistent data split
uniqueDates = np.array(sorted(df_rf['date'].unique()))
cut = int(len(uniqueDates) * 0.8)

trainDates = set(uniqueDates[:cut])
validDates = set(uniqueDates[cut:])

trainMask = df_rf['date'].isin(trainDates)
validMask = df_rf['date'].isin(validDates)

XTrain_rf = X_rf.loc[trainMask].copy()
XValid_rf = X_rf.loc[validMask].copy()
yTrain_rf = y_rf.loc[trainMask].copy()
yValid_rf = y_rf.loc[validMask].copy()

# 4) Preprocessing for RandomForest (one-hot encoding for categorical features)
XTrain_rf['asset_cat'] = XTrain_rf['asset_cat'].astype(str)
XValid_rf['asset_cat'] = XValid_rf['asset_cat'].astype(str)

XTrain_rf_enc = pd.get_dummies(XTrain_rf, columns=['asset_cat'], drop_first=False)
XValid_rf_enc = pd.get_dummies(XValid_rf, columns=['asset_cat'], drop_first=False)
XValid_rf_enc = XValid_rf_enc.reindex(columns=XTrain_rf_enc.columns, fill_value=0)

In [ ]:
# 5) Train RandomForest model
set_global_seed(GLOBAL_SEED)

model_rf = RandomForestClassifier(
    n_estimators=3000,
    criterion='gini',
    max_depth=14,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features='log2',
    max_samples=0.4,
    class_weight='balanced_subsample',
    bootstrap=True,
    random_state=GLOBAL_SEED,
    n_jobs=1
)


model_rf.fit(XTrain_rf_enc, yTrain_rf)

# 6) Predict probabilities on validation set
pValid_rf = model_rf.predict_proba(XValid_rf_enc)[:, 1]

# 7) Threshold tuning (optimize for Balanced Accuracy)
thrCandidates_rf = np.quantile(pValid_rf, np.linspace(0.05, 0.95, 181))

optThr_rf = 0.5
optBacc_rf = -np.inf

for thr in thrCandidates_rf:
    yHat = (pValid_rf >= thr).astype(int)
    bacc = balanced_accuracy_score(yValid_rf, yHat)
    if bacc > optBacc_rf:
        optBacc_rf = bacc
        optThr_rf = thr

yHat05_rf = (pValid_rf >= 0.5).astype(int)
yHatOptThr_rf = (pValid_rf >= optThr_rf).astype(int)

print(f"Valid AUC: {roc_auc_score(yValid_rf, pValid_rf):.4f}")
print("Balanced Accuracy 기준 최적 threshold:", round(float(optThr_rf), 4))
print("Balanced Accuracy:", round(optBacc_rf, 4))

# 8) Output results
report(yValid_rf, yHatOptThr_rf, pValid_rf, f'RandomForest Tuning Result')

### 2-3) MLP

In [ ]:
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

set_global_seed(GLOBAL_SEED)

# 1) Use the same labeledDf data and time split as previous models
df_mlp = labeledDf.copy()

df_mlp['date'] = pd.to_datetime(df_mlp['date'])
df_mlp = df_mlp.sort_values(['date', 'asset']).reset_index(drop=True)
df_mlp['asset_cat'] = df_mlp['asset'].astype('category')

idCols = {'asset', 'date'}
leakageCols = {'y', 'ret_1d_next', 'tau'}

safeFeatures_mlp = [c for c in featureCols if c not in leakageCols and c not in idCols]
safeFeatures_mlp = safeFeatures_mlp + ['asset_cat']

X_mlp = df_mlp[safeFeatures_mlp].copy()
y_mlp = df_mlp['y'].astype(int).copy()

uniqueDates = np.array(sorted(df_mlp['date'].unique()))
cut = int(len(uniqueDates) * 0.8)

trainDates = set(uniqueDates[:cut])
validDates = set(uniqueDates[cut:])

trainMask = df_mlp['date'].isin(trainDates)
validMask = df_mlp['date'].isin(validDates)

XTrain_mlp = X_mlp.loc[trainMask].copy()
XValid_mlp = X_mlp.loc[validMask].copy()
yTrain_mlp = y_mlp.loc[trainMask].copy()
yValid_mlp = y_mlp.loc[validMask].copy()

# 2) One-hot encode 'asset_cat', scale other features
XTrain_mlp['asset_cat'] = XTrain_mlp['asset_cat'].astype(str)
XValid_mlp['asset_cat'] = XValid_mlp['asset_cat'].astype(str)

XTrain_mlp = pd.get_dummies(XTrain_mlp, columns=['asset_cat'], drop_first=False)
XValid_mlp = pd.get_dummies(XValid_mlp, columns=['asset_cat'], drop_first=False)

XValid_mlp = XValid_mlp.reindex(columns=XTrain_mlp.columns, fill_value=0)

# 3) Scale features using StandardScaler fitted on training data only
scaler = StandardScaler()
XTrain_scaled = scaler.fit_transform(XTrain_mlp)
XValid_scaled = scaler.transform(XValid_mlp)

# 4) Convert to PyTorch Tensors and create DataLoaders
# CPU 고정: CUDA/cuDNN/BatchNorm/Dropout 조합의 미세 비결정성을 제거해 재현성 검증을 우선함.
device = torch.device('cpu')

XTrain_t = torch.tensor(XTrain_scaled, dtype=torch.float32)
yTrain_t = torch.tensor(yTrain_mlp.values, dtype=torch.float32).view(-1, 1)

XValid_t = torch.tensor(XValid_scaled, dtype=torch.float32)
yValid_t = torch.tensor(yValid_mlp.values, dtype=torch.float32).view(-1, 1)

trainDataset = TensorDataset(XTrain_t, yTrain_t)
validDataset = TensorDataset(XValid_t, yValid_t)

_train_gen = torch.Generator()
_train_gen.manual_seed(GLOBAL_SEED)
trainLoader = DataLoader(trainDataset, batch_size=128, shuffle=True, generator=_train_gen, num_workers=0)
validLoader = DataLoader(validDataset, batch_size=256, shuffle=False, num_workers=0)

# 5) Define MLP architecture
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

model_mlp = MLP(XTrain_t.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=1e-3, weight_decay=1e-4)

# 6) Train MLP model with early stopping
bestAuc = -np.inf
bestState = None
patience = 20
wait = 0
numEpochs = 200

for epoch in range(numEpochs):
    model_mlp.train()
    trainLossSum = 0.0

    for xb, yb in trainLoader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model_mlp(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        trainLossSum += loss.item() * xb.size(0)

    model_mlp.eval()
    validProbList = []

    with torch.no_grad():
        for xb, yb in validLoader:
            xb = xb.to(device)
            logits = model_mlp(xb)
            probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
            validProbList.append(probs)

    pValid_mlp = np.concatenate(validProbList)
    auc = roc_auc_score(yValid_mlp, pValid_mlp)

    print(f"Epoch {epoch+1:03d} | Train Loss: {trainLossSum / len(trainDataset):.6f} | Valid AUC: {auc:.6f}")

    if auc > bestAuc:
        bestAuc = auc
        bestState = copy.deepcopy(model_mlp.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping triggered.")
            break

# 7) Restore best model and predict on validation set
model_mlp.load_state_dict(bestState)
model_mlp.eval()

with torch.no_grad():
    pValid_mlp = torch.sigmoid(model_mlp(XValid_t.to(device))).squeeze(1).cpu().numpy()

print("MLP valid AUC:", roc_auc_score(yValid_mlp, pValid_mlp))

# 8) Threshold tuning
thrCandidates_mlp = np.quantile(pValid_mlp, np.linspace(0.05, 0.95, 181))

optThr_mlp = 0.5
optBacc_mlp = -np.inf

for thr in thrCandidates_mlp:
    yHat = (pValid_mlp >= thr).astype(int)
    bacc = balanced_accuracy_score(yValid_mlp, yHat)
    if bacc > optBacc_mlp:
        optBacc_mlp = bacc
        optThr_mlp = thr

yHat05_mlp = (pValid_mlp >= 0.5).astype(int)
yHatOptThr_mlp = (pValid_mlp >= optThr_mlp).astype(int)

print("Balanced Accuracy 기준 최적 threshold:", round(float(optThr_mlp), 4))
print("Balanced Accuracy:", round(optBacc_mlp, 4))

report(yValid_mlp, yHat05_mlp, pValid_mlp, "MLP 검증셋 at threshold : 0.5")
report(yValid_mlp, yHatOptThr_mlp, pValid_mlp, f"MLP 검증셋 at optimal threshold : {optThr_mlp:.4f}")

In [ ]:
# artifact save — Google Drive / c-index / artifacts 바로 아래에 저장
# 실행 위치: GBM/RF/MLP 학습과 threshold tuning이 끝난 직후

import json
import joblib
import torch
from pathlib import Path
from datetime import datetime

ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def _json_safe(x):
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    if hasattr(x, 'item'):
        return x.item()
    if isinstance(x, (list, tuple)):
        return [_json_safe(v) for v in x]
    if isinstance(x, dict):
        return {str(k): _json_safe(v) for k, v in x.items()}
    return str(x)

# 1) Models and preprocessing
model_gbm.save_model(str(ARTIFACT_DIR / 'model_gbm.txt'))
joblib.dump(model_rf, ARTIFACT_DIR / 'model_rf.pkl')
joblib.dump(scaler, ARTIFACT_DIR / 'scaler.pkl')

torch.save({
    'state_dict': model_mlp.state_dict(),
    'input_dim': int(XTrain_t.shape[1]),
    'x_columns': list(XTrain_mlp.columns),
    'model_class': 'MLP',
}, ARTIFACT_DIR / 'model_mlp_state.pt')

# 2) Metadata needed to reproduce prediction/evaluation without guessing paths
threshold_map = {
    'gbm': float(optThr),
    'rf': float(optThr_rf),
    'mlp': float(optThr_mlp),
}

model_metadata = {
    'saved_at': datetime.now().isoformat(timespec='seconds'),
    'artifact_dir': str(ARTIFACT_DIR),
    'global_seed': int(GLOBAL_SEED) if 'GLOBAL_SEED' in globals() else None,
    'threshold_map': threshold_map,
    'featureCols': list(featureCols) if 'featureCols' in globals() else None,
    'safeFeatures_gbm': list(safeFeatures) if 'safeFeatures' in globals() else None,
    'safeFeatures_rf': list(safeFeatures_rf) if 'safeFeatures_rf' in globals() else None,
    'safeFeatures_mlp': list(safeFeatures_mlp) if 'safeFeatures_mlp' in globals() else None,
    'rf_encoded_columns': list(XTrain_rf_enc.columns) if 'XTrain_rf_enc' in globals() else None,
    'mlp_encoded_columns': list(XTrain_mlp.columns) if 'XTrain_mlp' in globals() else None,
}

with open(ARTIFACT_DIR / 'model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(_json_safe(model_metadata), f, ensure_ascii=False, indent=2)

# 3) Early dataframes available at this stage
for _name in ['panelDf', 'labeledDf', 'df', 'df_rf', 'df_mlp']:
    if _name in globals():
        globals()[_name].to_pickle(ARTIFACT_DIR / f'{_name}_cache.pkl')

print('[artifact save complete]')
print('ARTIFACT_DIR =', ARTIFACT_DIR)
print('saved files:', sorted(p.name for p in ARTIFACT_DIR.iterdir() if p.is_file()))



In [ ]:
# artifact load — Google Drive / c-index / artifacts 바로 아래에서 로드

import json
import joblib
import torch
import lightgbm as lgb
from pathlib import Path

ARTIFACT_DIR = PROJECT_DIR / 'artifacts'

metadata_path = ARTIFACT_DIR / 'model_metadata.json'
if metadata_path.exists():
    with open(metadata_path, 'r', encoding='utf-8') as f:
        model_metadata = json.load(f)
    thrMap_saved = model_metadata.get('threshold_map', {})
else:
    model_metadata = {}
    thrMap_saved = {}

model_gbm = lgb.Booster(model_file=str(ARTIFACT_DIR / 'model_gbm.txt'))
model_rf = joblib.load(ARTIFACT_DIR / 'model_rf.pkl')
scaler = joblib.load(ARTIFACT_DIR / 'scaler.pkl')

_mlp_checkpoint = torch.load(ARTIFACT_DIR / 'model_mlp_state.pt', map_location=device)
_mlp_input_dim = int(_mlp_checkpoint.get('input_dim', XTrain_mlp.shape[1]))
model_mlp = MLP(input_dim=_mlp_input_dim).to(device)
model_mlp.load_state_dict(_mlp_checkpoint['state_dict'])
model_mlp.eval()

print('[artifact load complete]')
print('ARTIFACT_DIR =', ARTIFACT_DIR)
print('loaded thresholds:', thrMap_saved)



---

## <mark>  3) 설명 추출: C-index 입력으로서의 랭킹 설계 </mark>

### 3-1) 설명 추출의 목적

본 연구의 설명 추출 목적은 개별 예측에 대한 설명 정확성 검증이 아니라, C-index 산출에 사용되는 입력 데이터를 구성하는 데 있음. 즉, 동일 시점에서 서로 다른 설명 기준이 산출하는 중요도 랭킹을 수집하고, 이들 간 합의 수준을 정량화하기 위한 기반 정보를 생성하는 것이 핵심 목적임. 또한 C-index는 실제 거래 필터링에 활용되는 신뢰도 지표이므로, 설명 정보는 Train set이 아닌 Test set을 중심으로 산출함.

### 3-2) 설명 기법 선택

본 연구는 서로 다른 설명 관점을 갖는 복수의 XAI 기법 간 불일치 자체를 신뢰도 신호로 활용함.

* Permutation Importance  
* SHAP (TreeExplainer)  
* *LIME, Integrated Gradients 등은 시계열 환경에서의 불안정성 및 계산 비용 문제로 1차 실험에서는 제외*

### 3-3) Time-local 랭킹 산출 방식 (regime에 대한 판단)

* 시점 (t)를 포함하는 최근 (W)일 rolling window를 하나의 설명 단위로 설정  
* 각 window마다 다음 랭킹 산출  
  * **R₁(t)**: Permutation Importance 기반 중요도 랭킹  
  * **R₂(t**): SHAP 기반 중요도 랭킹
* 두 랭킹을 해당 시점의 time-local explanation으로 간주  

이 구조를 통해서 레짐 전환, 변동성 급증 등 불확실성 국면에서 설명 기법 간 합의 약화가 자연스럽게 발생하도록 설계되며, C-index를 "해당 시점의 예측을 신뢰할 수 있는가"의 문제로 설정할 수 있게 됨.


In [ ]:
# 1) Operational Evaluation Period (OEP) setup: same as validation set dates for explanation and reliability evaluation
dfAll = labeledDf.copy()
dfAll['date'] = pd.to_datetime(dfAll['date'])
dfAll['asset_cat'] = pd.Categorical(dfAll['asset'], categories=df['asset_cat'].cat.categories)
dfAll = dfAll.sort_values(['asset', 'date']).reset_index(drop=True)

oepEnd = dfAll['date'].max()
oepStart = oepEnd - pd.DateOffset(years=2)

oepDf = dfAll[
    (dfAll['date'] >= oepStart) &
    (dfAll['date'] <= oepEnd)
].copy()

oepDf = oepDf.sort_values(['asset', 'date']).reset_index(drop=True)

print("OEP period:", oepStart.date(), "->", oepEnd.date())
print("OEP rows:", len(oepDf), "| assets:", oepDf['asset'].nunique())

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import copy
import shap
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

W, STEP, TOPK, SEED = 60, 2, 10, 42
SHAP_SAMPLE = 30

set_global_seed(SEED)

# CPU 고정: MLP 학습/DeepExplainer 재현성 검증을 위해 GPU 비결정성을 배제함.
device = torch.device('cpu')

feature_universe = ['ret_1d','ret_5d','ret_10d','vol_20','vol_60','atr_14',
                    'ema_12','ema_26','macd','rsi_14','roc_10','volume_z20',
                    '^VIX','IEF','UUP']
universe_set = set(feature_universe)

thrMap = {
    'gbm': float(optThr),
    'rf': float(optThr_rf),
    'mlp': float(optThr_mlp),
}

rfInputCols = XTrain_rf_enc.columns.tolist()
mlpInputCols = XTrain_mlp.columns.tolist()

dfAll = labeledDf.copy()
dfAll['date'] = pd.to_datetime(dfAll['date'])
dfAll = dfAll.sort_values(['asset', 'date']).reset_index(drop=True)
dfAll['asset_cat'] = pd.Categorical(dfAll['asset'], categories=df['asset_cat'].cat.categories)

oepEnd = dfAll['date'].max()
oepStart = oepEnd - pd.DateOffset(years=2)

oepDf = dfAll[
    (dfAll['date'] >= oepStart) &
    (dfAll['date'] <= oepEnd)
].copy()

oepDf = oepDf.sort_values(['asset', 'date']).reset_index(drop=True)

bgSize = min(32, len(XTrain_scaled))
bgIdx = np.random.RandomState(SEED).choice(len(XTrain_scaled), size=bgSize, replace=False)
background_mlp = torch.tensor(XTrain_scaled[bgIdx], dtype=torch.float32).to(device)

explainerMap = {
    'gbm': shap.TreeExplainer(model_gbm),
    'rf': shap.TreeExplainer(model_rf),
    'mlp': shap.DeepExplainer(model_mlp, background_mlp)
}

def preprocess_for_model(model_name, Xraw):
    Xraw = Xraw.copy()

    if model_name == 'gbm':
        Xproc = Xraw.copy()
        Xproc['asset_cat'] = pd.Categorical(
            Xproc['asset_cat'],
            categories=dfAll['asset_cat'].cat.categories
        )
        return Xproc, list(Xproc.columns)

    elif model_name == 'rf':
        Xproc = Xraw.copy()
        Xproc['asset_cat'] = Xproc['asset_cat'].astype(str)
        Xproc = pd.get_dummies(Xproc, columns=['asset_cat'], drop_first=False)
        Xproc = Xproc.reindex(columns=rfInputCols, fill_value=0)
        return Xproc, list(Xproc.columns)

    elif model_name == 'mlp':
        Xproc = Xraw.copy()
        Xproc['asset_cat'] = Xproc['asset_cat'].astype(str)
        Xproc = pd.get_dummies(Xproc, columns=['asset_cat'], drop_first=False)
        Xproc = Xproc.reindex(columns=mlpInputCols, fill_value=0)
        Xscaled = scaler.transform(Xproc)
        return Xscaled, list(Xproc.columns)

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

def predict_proba_model(model_name, Xraw):
    Xproc, _ = preprocess_for_model(model_name, Xraw)

    if model_name == 'gbm':
        return model_gbm.predict(Xproc, num_iteration=model_gbm.best_iteration)

    elif model_name == 'rf':
        return model_rf.predict_proba(Xproc)[:, 1]

    elif model_name == 'mlp':
        Xt = torch.tensor(Xproc, dtype=torch.float32).to(device)
        model_mlp.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model_mlp(Xt)).squeeze(1).cpu().numpy()
        return probs

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

def permutation_importance_ranking(model_name, Xw, yw, thr, n_repeats=1, seed=42):
    rng = np.random.default_rng(seed)
    p_base = predict_proba_model(model_name, Xw)
    yhat_base = (p_base >= thr).astype(int)
    base_score = balanced_accuracy_score(yw, yhat_base)

    importances = {}
    for col in feature_universe:
        drops = []
        for _ in range(n_repeats):
            Xperm = Xw.copy()
            perm_idx = rng.permutation(len(Xperm))
            Xperm[col] = Xperm[col].to_numpy()[perm_idx]
            p_perm = predict_proba_model(model_name, Xperm)
            yhat_perm = (p_perm >= thr).astype(int)
            score = balanced_accuracy_score(yw, yhat_perm)
            drops.append(base_score - score)
        importances[col] = float(np.mean(drops))

    ranked = sorted(importances.items(), key=lambda x: x[1], reverse=True)
    rank_list = [k for k, _ in ranked if k in universe_set]
    return rank_list, importances

def shap_ranking(model_name, Xw):
    explainer = explainerMap[model_name]
    Xproc, proc_cols = preprocess_for_model(model_name, Xw)

    if model_name in ['gbm', 'rf']:
        Xshap = Xproc.iloc[:SHAP_SAMPLE] if len(Xproc) > SHAP_SAMPLE else Xproc
        shap_values = explainer.shap_values(Xshap, check_additivity=False)

        if isinstance(shap_values, list):
            values = shap_values[1] if len(shap_values) == 2 else shap_values[0]
        else:
            values = shap_values

        values = np.asarray(values)

        if values.ndim == 3:
            mean_abs = np.abs(values).mean(axis=(0, 2))
        elif values.ndim == 2:
            mean_abs = np.abs(values).mean(axis=0)
        else:
            mean_abs = np.abs(values).reshape(-1)

        raw_imp = dict(zip(proc_cols, mean_abs))

    elif model_name == 'mlp':
        Xshap = Xproc[:SHAP_SAMPLE] if len(Xproc) > SHAP_SAMPLE else Xproc
        Xt = torch.tensor(Xshap, dtype=torch.float32).to(device)
        shap_values = explainer.shap_values(Xt)

        if isinstance(shap_values, list):
            values = shap_values[0]
        else:
            values = shap_values

        values = np.asarray(values)

        if values.ndim == 3:
            if values.shape[-1] == 1:
                values = values.squeeze(-1)
            elif values.shape[1] == 1:
                values = values.squeeze(1)

        if values.ndim == 2:
            mean_abs = np.abs(values).mean(axis=0)
        else:
            mean_abs = np.abs(values).reshape(-1)

        raw_imp = dict(zip(proc_cols, mean_abs))

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    merged_imp = {}
    for f in feature_universe:
        if f in raw_imp:
            v = np.asarray(raw_imp[f])
            merged_imp[f] = float(v.mean())
        else:
            matched = [k for k in raw_imp.keys() if k.startswith(f + "_")]
            if len(matched) > 0:
                vals = [float(np.asarray(raw_imp[k]).mean()) for k in matched]
                merged_imp[f] = float(np.sum(vals))
            else:
                merged_imp[f] = 0.0

    ranked = sorted(merged_imp.items(), key=lambda x: x[1], reverse=True)
    rank_list = [k for k, _ in ranked]
    return rank_list, merged_imp

def collect_rankings(model_name, oep_df):
    records = []
    thr = thrMap[model_name]

    print(f"\n=== START {model_name.upper()} ===")

    for asset, g in oep_df.groupby('asset'):
        print(f"[{model_name}] asset: {asset} | rows: {len(g)}")

        g = g.sort_values('date').reset_index(drop=True)
        if len(g) < W:
            print(f"[{model_name}] skip {asset} (len={len(g)} < W={W})")
            continue

        Xg = g[feature_universe + ['asset_cat']].copy()
        yg = g['y'].astype(int).reset_index(drop=True)

        total_steps = len(range(W - 1, len(g), STEP))
        print(f"[{model_name}] {asset} total windows: {total_steps}")

        for i_idx, i in enumerate(range(W - 1, len(g), STEP), 1):
            if i_idx == 1 or i_idx % 5 == 0 or i_idx == total_steps:
                print(f"[{model_name}] {asset} window {i_idx}/{total_steps}")

            Xw = Xg.iloc[i - W + 1:i + 1].copy()
            yw = yg.iloc[i - W + 1:i + 1].copy()

            r_perm, _ = permutation_importance_ranking(
                model_name, Xw, yw, thr, n_repeats=1, seed=SEED
            )
            r_shap, _ = shap_ranking(model_name, Xw)

            records.append({
                'model': model_name,
                'asset': asset,
                'date': g.loc[i, 'date'],
                'permFullRank': r_perm,
                'shapFullRank': r_shap
            })

    print(f"=== END {model_name.upper()} | rows={len(records)} ===\n")
    return pd.DataFrame(records)


In [ ]:
print("GBM 시작")
rankings_gbm = collect_rankings('gbm', oepDf)

print("RF 시작")
rankings_rf = collect_rankings('rf', oepDf)

print("MLP 시작")
rankings_mlp = collect_rankings('mlp', oepDf)

rankingsDf = pd.concat(
    [rankings_gbm, rankings_rf, rankings_mlp],
    axis=0
).reset_index(drop=True)

print(rankingsDf.head())
print("rows:", len(rankingsDf))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

save_path = str(FIGURES_DIR)
os.makedirs(save_path, exist_ok=True)

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 9,
    'figure.titlesize': 14
})

df = rankingsDf.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['model', 'asset', 'date']).reset_index(drop=True)

Ks = [1, 3, 5, 10]

thetaByK = {
    1: 1.00,
    3: 2/3,
    5: 0.60,
    10: 0.60
}

def topk_overlap(a, b, k):
    a = list(a)[:k] if a is not None else []
    b = list(b)[:k] if b is not None else []
    if len(a) == 0 or len(b) == 0:
        return np.nan
    return len(set(a) & set(b)) / k

for k in Ks:
    df[f'overlap@{k}'] = df.apply(
        lambda r: topk_overlap(r['permFullRank'], r['shapFullRank'], k),
        axis=1
    )
    df[f'disagree@{k}'] = (df[f'overlap@{k}'] < thetaByK[k]).astype(float)

overlapSummary = (
    df.groupby('model')[[f'overlap@{k}' for k in Ks]]
    .mean()
    .round(4)
)

disagreeSummary = (
    df.groupby('model')[[f'disagree@{k}' for k in Ks]]
    .mean()
    .round(4)
)

assetDisagreeSummary = (
    df.groupby(['model', 'asset'])[[f'disagree@{k}' for k in Ks]]
    .mean()
    .round(4)
    .reset_index()
)

print("=== Mean Overlap by Model ===")
display(overlapSummary)

print("=== Disagreement Rate by Model ===")
display(disagreeSummary)

print("=== Disagreement Rate by Model x Asset ===")
display(assetDisagreeSummary)

dateModelDisagree = (
    df.groupby(['date', 'model'])[[f'disagree@{k}' for k in Ks]]
    .mean()
    .reset_index()
    .sort_values(['model', 'date'])
)

dateModelOverlap = (
    df.groupby(['date', 'model'])[[f'overlap@{k}' for k in Ks]]
    .mean()
    .reset_index()
    .sort_values(['model', 'date'])
)

ROLL = 5

for k in Ks:
    dateModelDisagree[f'disagree@{k}_roll'] = (
        dateModelDisagree.groupby('model')[f'disagree@{k}']
        .transform(lambda s: s.rolling(ROLL, min_periods=1).mean())
    )
    dateModelOverlap[f'overlap@{k}_roll'] = (
        dateModelOverlap.groupby('model')[f'overlap@{k}']
        .transform(lambda s: s.rolling(ROLL, min_periods=1).mean())
    )

fig1, axes = plt.subplots(2, 2, figsize=(16, 12))
fig1.suptitle("C-index Input Diagnostics: Permutation vs SHAP Agreement", fontsize=14)

ax = axes[0, 0]
x = np.arange(len(Ks))
models = overlapSummary.index.tolist()
width = 0.8 / max(len(models), 1)
for i, model in enumerate(models):
    vals = [overlapSummary.loc[model, f'overlap@{k}'] for k in Ks]
    ax.bar(x + i * width, vals, width=width, label=model)
ax.set_title("Mean Overlap by Model")
ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels([f"Top-{k}" for k in Ks])
ax.set_ylabel("overlap")
ax.set_ylim(0, 1)
ax.legend()

ax = axes[0, 1]
for i, model in enumerate(models):
    vals = [disagreeSummary.loc[model, f'disagree@{k}'] for k in Ks]
    ax.bar(x + i * width, vals, width=width, label=model)
ax.set_title("Disagreement Rate by Model")
ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels([f"Top-{k}" for k in Ks])
ax.set_ylabel("rate")
ax.set_ylim(0, 1)
ax.legend()

ax = axes[1, 0]
for model in sorted(dateModelDisagree['model'].unique()):
    sub = dateModelDisagree[dateModelDisagree['model'] == model]
    ax.plot(sub['date'], sub['disagree@5_roll'], label=model)
ax.set_title(f"Rolling Disagreement Over Time (Top-5, window={ROLL})")
ax.set_ylabel("rate")
ax.set_ylim(0, 1)
ax.legend()

ax = axes[1, 1]
for model in sorted(dateModelOverlap['model'].unique()):
    sub = dateModelOverlap[dateModelOverlap['model'] == model]
    ax.plot(sub['date'], sub['overlap@5_roll'], label=model)
ax.set_title(f"Rolling Overlap Over Time (Top-5, window={ROLL})")
ax.set_ylabel("overlap")
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout(pad=3.0)
plt.show()

assetTop5 = (
    df.groupby(['model', 'asset'])['disagree@5']
    .mean()
    .reset_index()
)

fig2, ax = plt.subplots(figsize=(12, 8))
models = sorted(assetTop5['model'].unique())
assets = sorted(assetTop5['asset'].unique())
x = np.arange(len(assets))
width = 0.8 / max(len(models), 1)

for i, model in enumerate(models):
    sub = assetTop5[assetTop5['model'] == model].set_index('asset').reindex(assets)
    ax.bar(x + i * width, sub['disagree@5'].values, width=width, label=model)

ax.set_title("Top-5 Disagreement Rate by Asset")
ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels(assets)
ax.set_ylabel("rate")
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout(pad=3.0)
plt.show()

assetDateTop5 = (
    df.groupby(['date', 'model', 'asset'])['disagree@5']
    .mean()
    .reset_index()
    .sort_values(['asset', 'model', 'date'])
)

assetDateTop5['disagree@5_roll'] = (
    assetDateTop5.groupby(['asset', 'model'])['disagree@5']
    .transform(lambda s: s.rolling(ROLL, min_periods=1).mean())
)

assets = sorted(assetDateTop5['asset'].unique())

for asset in assets:
    fig3, ax = plt.subplots(figsize=(16, 6))
    subAsset = assetDateTop5[assetDateTop5['asset'] == asset]

    for model in sorted(subAsset['model'].unique()):
        sub = subAsset[subAsset['model'] == model]
        ax.plot(sub['date'], sub['disagree@5_roll'], label=model)

    ax.set_title(f"{asset}: Rolling Disagreement (Top-5)")
    ax.set_ylabel("rate")
    ax.set_ylim(0, 1)
    ax.legend()

    plt.tight_layout(pad=3.0)
    plt.show()

In [ ]:
import os

save_path = str(FIGURES_DIR)
os.makedirs(save_path, exist_ok=True)

# ===== TABLE SAVE (with index) =====
def save_df_as_image(df, filename, dpi=450, title=None):
    import pandas as pd
    import matplotlib.pyplot as plt

    df_show = df.copy()

    if isinstance(df_show, pd.Series):
        df_show = df_show.to_frame()

    index_name = df_show.index.name if df_show.index.name is not None else "model"
    df_show = df_show.reset_index().rename(columns={df_show.columns[0]: index_name})

    nrows, ncols = df_show.shape
    fig_w = max(8, ncols * 2.2)
    fig_h = max(2.2, (nrows + 1) * 0.55)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")

    if title:
        ax.set_title(title, fontsize=12, pad=12)

    tbl = ax.table(
        cellText=df_show.values,
        colLabels=df_show.columns,
        loc="center",
        cellLoc="center"
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.05, 1.55)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(1.0)
        if r == 0:
            cell.set_text_props(weight="bold")
        if c == 0:
            cell.set_text_props(weight="bold")

    plt.tight_layout(pad=1.5)
    fig.savefig(os.path.join(save_path, filename), dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)


# ===== TABLE SAVE (NO INDEX - asset table only) =====
def save_df_no_index(df, filename, dpi=450, title=None):
    import matplotlib.pyplot as plt

    df_show = df.copy()

    nrows, ncols = df_show.shape
    fig_w = max(8, ncols * 2.2)
    fig_h = max(2.2, (nrows + 1) * 0.55)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")

    if title:
        ax.set_title(title, fontsize=12, pad=12)

    tbl = ax.table(
        cellText=df_show.values,
        colLabels=df_show.columns,
        loc="center",
        cellLoc="center"
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.05, 1.55)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(1.0)
        if r == 0:
            cell.set_text_props(weight="bold")

    plt.tight_layout(pad=1.5)
    fig.savefig(os.path.join(save_path, filename), dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)


# ===== TABLE SAVE 실행 =====
save_df_as_image(overlapSummary, "overlap_summary.png", dpi=450,
                 title="Mean Overlap by Model")

save_df_as_image(disagreeSummary, "disagree_summary.png", dpi=450,
                 title="Disagreement Rate by Model")

save_df_no_index(assetDisagreeSummary, "asset_disagree_summary.png", dpi=450,
                 title="Disagreement Rate by Model x Asset")


# ===== FIGURE SAVE =====
fig1.savefig(os.path.join(save_path, "figure1_overall.png"),
             dpi=450, bbox_inches='tight')

fig2.savefig(os.path.join(save_path, "figure2_asset.png"),
             dpi=450, bbox_inches='tight')


# ===== FIGURE3 (asset별) SAVE =====
for asset in assets:
    fig3, ax = plt.subplots(figsize=(16, 6))

    subAsset = assetDateTop5[assetDateTop5['asset'] == asset]

    for model in sorted(subAsset['model'].unique()):
        sub = subAsset[subAsset['model'] == model]
        ax.plot(sub['date'], sub['disagree@5_roll'], label=model)

    ax.set_title(f"{asset}: Rolling Disagreement (Top-5)")
    ax.set_ylim(0, 1)
    ax.legend()

    plt.tight_layout()

    fig3.savefig(os.path.join(save_path, f"figure3_{asset}.png"),
                 dpi=450, bbox_inches='tight')

    plt.close(fig3)


print("saved all (450 dpi)")

### 3-5) 결과 해석 및 C-Index 필요성 확인

Permutation Importance와 SHAP 기반 랭킹은 동일한 모델과 동일한 데이터 window를 사용하였음에도 불구하고 상위 피처 구성에서 구조적으로 일관된 차이를 보인다. 이는 동일한 예측 결과에 대해서도 설명 기법에 따라 "어떤 근거를 중요하게 보는가"가 달라질 수 있음을 의미한다.

실행 결과에서도 Top-K가 작을수록 설명 불일치가 크게 나타났다. Top-3 disagreement는 GBM 0.8621, RF 0.8793, MLP 0.7586으로 모두 높은 수준이며, 이는 핵심 피처 수준에서 설명 기법 간 합의가 충분하지 않음을 보여준다. 반면 K가 10으로 커지면 disagreement가 GBM 0.1207, RF 0.1034, MLP 0.0345로 낮아지는데, 이는 설명 합의가 실질적으로 개선되었다기보다 Top-K 범위가 넓어지면서 기계적으로 중복이 증가한 결과로 해석하는 것이 타당하다.

따라서 본 결과는 AUC, Accuracy, Balanced Accuracy와 같은 전통적 예측 성능 지표만으로는 모델 판단 근거의 안정성을 식별하기 어렵다는 점을 뒷받침한다. 특히 base model의 AUC가 0.51~0.54 수준으로 약한 상황에서, 예측 확률 자체보다 **해당 예측을 구성한 설명 구조가 안정적인가**를 별도 지표로 점검할 필요가 있다.

C-index는 이러한 설명 기법 간 합의 수준을 기반으로 특정 시점의 모델 판단 구조적 안정성을 정량화한 지표이다. 다만 본 단계의 결과는 C-index 도입의 필요성을 보여주는 것이며, C-index 필터링이 실제 거래 성과를 일관되게 개선한다는 결론은 이후 성과평가 결과와 통계 검정을 통해 별도로 확인해야 한다.



### 3-6) 랭킹 데이터셋 구성
본 단계는 C-index 정의 옵션(Kendall–τ, RBO, Mallows)에 관계없이 일관된 계산이 가능하도록, 시점별 설명 정보를 범용 입력 형태로 저장하는 것을 목표로 하되, 다중 모델 환경(gbm, rf, mlp)에서도 해석적 정합성을 유지하도록 구조를 재설계함.

* 기본 단위 및 인덱싱

  * **분석 단위**: 모델 (m) × 자산 (a) × 시점 (t)
  * **윈도우 설정**: (W:60)을 하나의 설명 단위로 정의하고, (STEP:5) 간격으로 시점을 샘플링함
  * **설명 산출 원칙**: 동일한 모델 (m) 및 동일한 window에서 Permutation / SHAP 두 설명기법을 병렬 적용하여 랭킹을 산출함

  → 이를 통해 설명 불일치는 “설명기법 간 차이”로 해석 가능하도록 통제함 (모델 구조 차이와 분리)

* Feature 고정 / 해석 대상 피처 정의

  * 시점별 Top-K 변동으로 인한 비교 불가능성을 방지하기 위해 전체 feature universe를 고정함
  * `asset_cat`은 기본 분석에서 제외하여 모델 간 표현 차이(encoding 방식)에 따른 왜곡 방지
  * feature index mapping (`feature_to_id`)을 유지하여 거리 계산 및 확률모형 입력 안정성 확보

* 저장 포맷 (모델별 독립 저장)

각 (m, a, t) 단위에서 아래 정보를 저장하며, 모델 간 데이터는 결합하지 않고 독립적으로 유지함.

* **Full rank (핵심 입력)**

  * `permFullRank_m`, `shapFullRank_m`
  * 전체 피처 순위 정보를 보존하여 Kendall–τ, Mallows 계산에 직접 사용

* **Top-K list (보조 입력)**

  * `permTopK_m`, `shapTopK_m`
  * Top-3 / Top-5 / Top-10
  * overlap, RBO 등 상위 중요도 기반 지표 계산에 활용

* **Rank vector with ties (τ-b용 파생 표현)**

  * `permTiesK_m`, `shapTiesK_m`
  * Top-K 밖 피처를 공동 꼴찌(K+1)로 묶어 전체 순위 공간 유지
  * 시점 간 비교 가능성과 수학적 정합성 동시 확보

* 다중 모델 환경에서의 처리 전략

  * C-index는 **모델별로 독립적으로 계산**
    → ( C^{(gbm)}(t), C^{(rf)}(t), C^{(mlp)}(t) )

  * 모델 간 비교는 사후 분석 단계에서 수행
    → “어떤 모델이 더 안정적인 설명 구조를 갖는가”를 비교하는 용도로 활용

  * 단일 C-index로 모델을 혼합하는 방식은 사용하지 않음
    → 설명 불일치 원인이 “모델 구조 vs 설명기법 차이”로 분리되지 않기 때문

* 설계 의의

본 재구성은 기존 single-model 기반 설계를 다중 모델 환경으로 확장하면서도, C-index의 정의를 훼손하지 않도록 하기 위한 최소한의 구조적 수정임. 이를 통해 C-index는 여전히 “설명기법 간 합의도”를 측정하는 순수한 지표로 유지되며, 동시에 모델 간 robustness 비교까지 가능해지는 확장성을 확보함.







In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray",
    category=UserWarning
)

import shap
import copy
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import balanced_accuracy_score

# 0) Configuration Parameters
W = 60
STEP = 2
TOPK_LIST = [3, 5, 10]
SEED = 42

set_global_seed(SEED)
# CPU 고정: 실행 간 미세한 GPU 비결정성을 제거함.
device = torch.device('cpu')

# Feature columns definition (copied for robustness as it was missing from this cell's scope)
featureCols = ['ret_1d','ret_5d','ret_10d','vol_20','vol_60','atr_14','ema_12','ema_26','macd','rsi_14','roc_10','volume_z20'] + ['^VIX', 'IEF', 'UUP']

# Common feature universe for explanation comparison (excluding 'asset_cat' due to varying encoding)
feature_universe = [c for c in featureCols if c != 'asset_cat']
universe_set = set(feature_universe)

# Model-specific thresholds
thrMap = {
    'gbm': float(optThr),
    'rf': float(optThr_rf),
    'mlp': float(optThr_mlp),
}

# Input columns for RF and MLP models
rfInputCols = XTrain_rf_enc.columns.tolist()
mlpInputCols = XTrain_mlp.columns.tolist()

# 1) Prepare OEP Data
dfAll = labeledDf.copy()
dfAll['date'] = pd.to_datetime(dfAll['date'])
dfAll = dfAll.sort_values(['asset', 'date']).reset_index(drop=True)

if 'asset_cat' not in dfAll.columns:
    dfAll['asset_cat'] = dfAll['asset'].astype('category')
else:
    dfAll['asset_cat'] = pd.Categorical(
        dfAll['asset_cat'],
        categories=df['asset_cat'].cat.categories # Ensure categories are consistent
    )

oepEnd = dfAll['date'].max()
oepStart = oepEnd - pd.DateOffset(years=2)

oepDf = dfAll[
    (dfAll['date'] >= oepStart) &
    (dfAll['date'] <= oepEnd)
].copy()

oepDf = oepDf.sort_values(['asset', 'date']).reset_index(drop=True)

print("OEP period:", oepStart.date(), "->", oepEnd.date())
print("OEP rows:", len(oepDf), "| assets:", oepDf['asset'].nunique())

# 2) SHAP background for MLP
bgSize = min(32, len(XTrain_scaled))
bgIdx = np.random.RandomState(SEED).choice(len(XTrain_scaled), size=bgSize, replace=False)
background_mlp = torch.tensor(XTrain_scaled[bgIdx], dtype=torch.float32).to(device)

explainerMap = {
    'gbm': shap.TreeExplainer(model_gbm),
    'rf': shap.TreeExplainer(model_rf),
    'mlp': shap.DeepExplainer(model_mlp, background_mlp)
}

# 3) Utility Functions
def preprocess_for_model(model_name, Xraw):
    Xraw = Xraw.copy()

    if model_name == 'gbm':
        Xproc = Xraw.copy()
        if 'asset_cat' in Xproc.columns:
            # Ensure categorical type consistency for GBM
            Xproc['asset_cat'] = pd.Categorical(
                Xproc['asset_cat'],
                categories=dfAll['asset_cat'].cat.categories # Use global categories
            )
        return Xproc, list(Xproc.columns)

    elif model_name == 'rf':
        Xproc = Xraw.copy()
        Xproc['asset_cat'] = Xproc['asset_cat'].astype(str)
        Xproc = pd.get_dummies(Xproc, columns=['asset_cat'], drop_first=False)
        Xproc = Xproc.reindex(columns=rfInputCols, fill_value=0)
        return Xproc, list(Xproc.columns)

    elif model_name == 'mlp':
        Xproc = Xraw.copy()
        Xproc['asset_cat'] = Xproc['asset_cat'].astype(str)
        Xproc = pd.get_dummies(Xproc, columns=['asset_cat'], drop_first=False)
        Xproc = Xproc.reindex(columns=mlpInputCols, fill_value=0)
        Xscaled = scaler.transform(Xproc)
        return Xscaled, list(Xproc.columns)

    else:
        raise ValueError(f"Unknown model_name: {model_name}")


def predict_proba_model(model_name, Xraw):
    Xproc, _ = preprocess_for_model(model_name, Xraw)

    if model_name == 'gbm':
        return model_gbm.predict(Xproc, num_iteration=model_gbm.best_iteration)

    elif model_name == 'rf':
        return model_rf.predict_proba(Xproc)[:, 1]

    elif model_name == 'mlp':
        Xt = torch.tensor(Xproc, dtype=torch.float32).to(device)
        model_mlp.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model_mlp(Xt)).squeeze(1).cpu().numpy()
        return probs

    else:
        raise ValueError(f"Unknown model_name: {model_name}")


def permutation_importance_ranking(model_name, Xw, yw, thr, n_repeats=1, seed=42):
    rng = np.random.default_rng(seed)

    p_base = predict_proba_model(model_name, Xw)
    yhat_base = (p_base >= thr).astype(int)
    base_score = balanced_accuracy_score(yw, yhat_base)

    importances = {}

    # Use feature_universe for permutation importance, excluding 'asset_cat'
    features_to_permute = [f for f in feature_universe if f != 'asset_cat']

    for col in features_to_permute:
        drops = []
        for _ in range(n_repeats):
            Xperm = Xw.copy()
            perm_idx = rng.permutation(len(Xperm))
            Xperm[col] = Xperm[col].to_numpy()[perm_idx]

            p_perm = predict_proba_model(model_name, Xperm)
            yhat_perm = (p_perm >= thr).astype(int)
            score = balanced_accuracy_score(yw, yhat_perm)

            drops.append(base_score - score)

        importances[col] = float(np.mean(drops))

    # Filter to only include features in feature_universe for ranking
    ranked_filtered = [(k, v) for k, v in sorted(importances.items(), key=lambda x: x[1], reverse=True) if k in universe_set]
    rank_list = [k for k, _ in ranked_filtered]
    return rank_list, importances


SHAP_SAMPLE = 30

def shap_ranking(model_name, Xw):
    explainer = explainerMap[model_name]
    Xproc, proc_cols = preprocess_for_model(model_name, Xw)

    if model_name in ['gbm', 'rf']:
        Xshap = Xproc.iloc[:SHAP_SAMPLE] if len(Xproc) > SHAP_SAMPLE else Xproc
        shap_values = explainer.shap_values(Xshap, check_additivity=False)

        if isinstance(shap_values, list):
            values = shap_values[1] if len(shap_values) == 2 else shap_values[0]
        else:
            values = shap_values

    elif model_name == 'mlp':
        Xproc_s = Xproc[:SHAP_SAMPLE] if len(Xproc) > SHAP_SAMPLE else Xproc
        Xt = torch.tensor(Xproc_s, dtype=torch.float32).to(device)
        shap_values = explainer.shap_values(Xt)

        if isinstance(shap_values, list):
            values = shap_values[0]
        else:
            values = shap_values

        if isinstance(values, torch.Tensor):
            values = values.detach().cpu().numpy()

    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    values = np.array(values)

    if values.ndim == 3:
        if values.shape[-1] == 1:
            values = values.squeeze(-1)
        elif values.shape[-1] == 2:
            values = values[:, :, 1] # Assuming binary classification, take positive class
        else:
            raise ValueError(f"Unexpected SHAP shape: {values.shape}")
    elif values.ndim == 1 and model_name == 'gbm': # Handle single output for GBM (e.g., if shap_values is not a list for binary)
        values = values.reshape(1, -1) # Reshape to 2D for mean_abs

    mean_abs = np.abs(values).mean(axis=0)
    imp_series = pd.Series(mean_abs, index=proc_cols)

    # Filter to only include features in feature_universe
    imp_series = imp_series[[c for c in imp_series.index if c in universe_set]]

    ranked = imp_series.sort_values(ascending=False)
    return ranked.index.tolist(), ranked.to_dict()


def make_ties_rankvec(full_rank_feats, K, universe):
    pos = {f: i for i, f in enumerate(universe)}
    vec = np.full(len(universe), K + 1, dtype=int)

    for r, f in enumerate(full_rank_feats[:K], start=1):
        if f in pos: # Ensure feature exists in universe mapping
            vec[pos[f]] = r

    return vec.tolist()

# 4) Collect Rankings (Model-wise Independent)
def collect_rankings_modelwise(model_name, oep_df):
    records = []
    thr = thrMap[model_name]

    for asset, g in oep_df.groupby('asset'):
        g = g.sort_values('date').reset_index(drop=True)

        if len(g) < W:
            continue

        Xg = g[feature_universe + ['asset_cat']].copy()
        yg = g['y'].astype(int).copy()

        # Ensure asset_cat is handled consistently as a Categorical type with defined categories for GBM
        if model_name == 'gbm':
            Xg['asset_cat'] = pd.Categorical(
                Xg['asset_cat'],
                categories=dfAll['asset_cat'].cat.categories # Use global categories
            )

        for i in range(W - 1, len(g), STEP):
            Xw = Xg.iloc[i - W + 1:i + 1].copy()
            yw = yg.iloc[i - W + 1:i + 1].copy()

            # Permutation Importance
            r_perm_full, perm_imp = permutation_importance_ranking(
                model_name=model_name,
                Xw=Xw,
                yw=yw,
                thr=thr,
                n_repeats=1,
                seed=SEED
            )

            # SHAP Ranking
            r_shap_full, shap_imp = shap_ranking(
                model_name=model_name,
                Xw=Xw
            )

            row = {
                'model': model_name,
                'asset': asset,
                'date': g.loc[i, 'date'],
                'W': W,
                'STEP': STEP,
                'thr': thr,
                'permFullRank': r_perm_full,
                'shapFullRank': r_shap_full,
                'permImportances': perm_imp,
                'shapImportances': shap_imp,
            }

            for K in TOPK_LIST:
                row[f'permTop{K}'] = r_perm_full[:K]
                row[f'shapTop{K}'] = r_shap_full[:K]
                row[f'permTies{K}'] = make_ties_rankvec(r_perm_full, K, feature_universe)
                row[f'shapTies{K}'] = make_ties_rankvec(r_shap_full, K, feature_universe)

            records.append(row)

    out = (
        pd.DataFrame(records)
        .sort_values(['model', 'asset', 'date'])
        .reset_index(drop=True)
    )

    print(f"[{model_name}] rankings shape:", out.shape)
    return out

# 5) Execute: Calculate rankings for each model independently and concatenate
rankings_gbm = collect_rankings_modelwise('gbm', oepDf)
print("Columns of rankings_gbm:", rankings_gbm.columns.tolist()) # Debug print
rankings_rf  = collect_rankings_modelwise('rf',  oepDf)
print("Columns of rankings_rf:", rankings_rf.columns.tolist()) # Debug print
rankings_mlp = collect_rankings_modelwise('mlp', oepDf)
print("Columns of rankings_mlp:", rankings_mlp.columns.tolist()) # Debug print

rankingsDf = pd.concat(
    [rankings_gbm, rankings_rf, rankings_mlp],
    axis=0
).reset_index(drop=True)

print("\nColumns of concatenated rankingsDf:", rankingsDf.columns.tolist()) # Debug print
print("rankingsDf info:")
rankingsDf.info() # Debug print

print("rankings_gbm:", rankings_gbm.shape)
display(rankings_gbm.head(3))

print("rankings_rf:", rankings_rf.shape)
display(rankings_rf.head(3))

print("rankings_mlp:", rankings_mlp.shape)
display(rankings_mlp.head(3))

print("rankingsDf:", rankingsDf.shape)
display(rankingsDf.head(10))

# Preserve common feature universe
featureMap = {
    'feature_universe': feature_universe
}


### 3-7) 데이터셋 요약

| 관점        | 설명                                                                                                                                                            |
| --------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **시간**    | *Event-conditioned, rolling-window*<br>각 설명은 모델 (m) × 자산 (a) × 시점 (t) 단위로 정의됨. 시점 (t)를 끝으로 하는 길이 (W)의 rolling window에서 산출되며, STEP 간격으로 시점을 샘플링하여 시간적 중복을 제어함. |
| **피처 공간** | *고정 universe*<br>자산·시점과 무관하게 동일한 feature universe를 사용함. 모델 간 비교 가능성과 순위 기반 거리 계산의 정합성을 위해 공통 feature 집합을 유지하며, encoding 차이에 따른 왜곡을 제거함.                       |
| **설명기법**  | *Permutation (BA-drop), SHAP*<br>각 모델 내부에서 동일한 window에 대해 두 설명기법을 병렬 적용하여 랭킹을 산출함. 즉, 설명 합의는 모델 간이 아니라 모델 내부에서 정의되며, 설명 불일치는 순수하게 설명기법 간 차이로 해석 가능하도록 통제됨.    |
| **저장 수준** | *Full rank + Ties (모델별 독립 저장)*<br>각 (m,a,t) 단위에서 전체 랭킹을 저장하고, Top-K 및 tie-aware rank vector를 파생 생성함. 데이터는 모델별로 독립 저장되며, C-index 역시 모델별로 개별 산출됨.               |
| **확장성**   | *Kendall / RBO / Mallows 전부 대응*<br>Full rank 기반으로 Kendall–τ, Mallows 계산 가능하며, prefix 기반으로 RBO 및 overlap 계산 가능함. 동일 데이터셋으로 다양한 합의 정의를 일관되게 평가할 수 있음.           |

---

### Top-K 리스트가 불필요한 이유 (수정)

본 연구의 핵심 목적은 중요한 피처를 선별하는 것이 아니라, 동일한 모델과 동일한 시점에서 설명 기법 간 판단 구조의 합의 안정성을 정량적으로 평가하는 데 있음. 이때 핵심 전제는 모든 시점에서 비교 대상이 되는 피처 공간의 동일성 유지임. 다만, Top-K 자체가 완전히 불필요한 것은 아니며, 상위 중요 피처에 대한 강조 정보 보존을 위한 보조적 표현으로의 활용 가능성 존재함. 본 연구에서는 이를 위해 Top-K를 직접 리스트로 사용하는 대신, 전체 순위 공간을 유지한 상태에서 상위 K개를 강조하는 tie-aware rank vector를 사용함. 해당 방식은 Top-K의 해석적 직관 유지와 동시에 시점 간 비교 가능성 및 수학적 정합성 확보라는 장점 가짐.결론적으로, Top-K 리스트는 분석의 핵심 입력이 아닌 파생적 보조 표현으로 제한적으로 활용되며, C-index 산출의 중심은 full rank 기반 표현에 둠.





## <mark> 4) C-index 정의 및 정당성 논리 </mark>

### 4-1) C-index 소개

* 정의  
C-index는 **각 모델 (m)에 대해 독립적으로**, 동일한 모델, 동일한 데이터 구간, 동일한 시점(또는 동일한 rolling window)에서 산출된 서로 다른 설명 기법(XAI) 기반 피처 중요도 랭킹들 간의 합의(consensus) 수준을 정량화한 지표임.

* 해석  
  * C-index가 높은 경우:  
    동일 모델 내부에서 서로 다른 설명 기준에도 불구하고 중요하게 식별되는 피처가 일관되게 유지되는 상태로, 해당 모델의 결정 근거가 구조적으로 안정적인 국면임을 의미함.

  * C-index가 낮은 경우:  
    동일 모델 내부에서도 설명 기법에 따라 중요 피처가 상이하게 나타나는 상태로, 모델의 결정 근거가 특정 피처 선택이나 우연적 노이즈에 민감하게 흔들리는 국면임을 의미함. 이는 레짐 전환, 비정상성, 시장 불확실성 확대 구간과 연관될 가능성이 높음.


### 4-2) C-index를 신뢰도 지표로 해석하는 정당성

금융 시계열 데이터는 **비정상성과 레짐 변화가 빈번한 특성**을 가지며, 이로 인해 동일한 예측 확률을 출력하더라도 시점별 결정 근거는 크게 달라질 수 있음. 본 연구에서는 각 모델(gbm, rf, mlp)에 대해 **설명기법 간 합의를 모델 내부에서 독립적으로 평가**하므로, 설명 결과 간 불일치는 모델 구조 차이가 아닌 설명 관점 차이에서 기인한 것으로 해석 가능함.

서로 다른 XAI 기법들은 모델 내부 구조 기반 기여도, 예측 성능 민감도 등 **상이한 설명 관점**을 제공하므로 일정 수준의 불일치는 자연스럽게 발생함. 그럼에도 불구하고 동일 모델 내부에서 상위 피처 랭킹이 일관되게 합의되는 경우, 모델 판단이 우연적 노이즈나 특정 설명 기법에 과도하게 의존하지 않고 **안정적인 구조적 신호에 기반**했을 가능성이 높음.

반대로 동일 모델 내에서도 설명 기법 간 합의가 약화되는 시점은 동일한 예측이라 하더라도 내부 근거의 불안정성을 직접적으로 시사함. 이 경우 예측 확률이 높더라도 신뢰 가능한 신호로 해석하기 어려움.

따라서 C-index는 예측 확률(p)과는 독립적으로, **각 모델 내부에서 특정 시점의 판단 근거가 얼마나 안정적인지를 정량화하는 지표**이며, “해당 모델의 해당 시점 예측을 신뢰할 수 있는가”를 판단하기 위한 메타 수준의 신뢰도 지표로 해석 가능함.


### 4-3) C-index 정의 방식: 설명 랭킹 합의의 정량화 옵션

C-index는 **각 모델 (m)에 대해 독립적으로 계산되며**, 시점별 합의 수준을 정량화함.

C-index의 해석적 직관성과 설명 기법 간 결과가 완전히 상반되는 경우가 드물다는 경험적 특성을 고려하여, C-index의 <font color="red">**범위는 0에서 1**</font>로 정의하며 <font color="red">**1에 가까울수록 설명 랭킹 간 합의 수준이 높음**</font>을 의미하도록 공통 규칙을 설정함.

#### **Option A**: Kendall–tau based Consensus

* **Intuition**  
설명 랭킹 간 consensus를 상관계수 관점에서 정의함. 다만, 음의 상관은 반대 합의로 해석하지 않으며, 설명 간 불일치로 간주하여 합의 수준 0으로 처리함.

* **Definition**  
시점 $t$에서 두 설명 랭킹을 $R_1^{(m)}(t)$, $R_2^{(m)}(t)$라 할 때,
$$
C_{\tau}^{(m)}(t)
= \max\!\left(0,\; \tau_b\!\left(R_1^{(m)}(t), R_2^{(m)}(t)\right)\right)
$$

$$
\tau_b
=
\frac{C - D}
{\sqrt{(C + D + T_x)(C + D + T_y)}}
$$

* **Scale 특성**
$$
C_{\tau}^{(m)}(t) \in [0, 1]
$$

#### **Option B**: RBO-based Consensus

* **Intuition**  
모델 판단의 신뢰성은 상위 핵심 근거(Top-K)에 의해 주로 결정된다는 가정에 기반함.

* **Definition**  
시점 $t$에서 두 설명 랭킹 $R_1^{(m)}(t), R_2^{(m)}(t)$에 대해,
$$
C_{\mathrm{RBO}}^{(m)}(t)
=
\frac{(1-p)\sum_{d=1}^{K}p^{d-1}A_d(t)}{1-p^{K}}
$$

* **Scale**
$$
C_{\mathrm{RBO}}^{(m)}(t) \in [0,1]
$$

#### **Option C**: Mallows-model based Consensus

* **Intuition**  
설명 랭킹들이 하나의 중심 구조 주변에 얼마나 집중되어 있는지를 기반으로 합의 수준을 평가함.

* **Definition**  
$$
C_{\mathrm{M}}^{(m)}(t)
=
\exp\!\left( - \hat{\phi}^{(m)}(t) \right)
$$

* **Scale**
$$
C_{\mathrm{M}}^{(m)}(t) \in (0,1]
$$



### 4-4) C-index 옵션별 입력 조합 구조 및 최종 비교 설계

모든 C-index는 **모델별로 독립적으로 계산되며**, 이후 비교는 사후 분석 단계에서 수행함.

Option A, B, C 각각의 수학적 정의에 부합하는 입력 조합만을 선택하여 비교 대상으로 설정하며, 불필요한 조합 확장은 배제함.


### 4-5) 구성할 C-index 세트

각 시점 \((m,a,t)\)에서 다음과 같은 C-index 세트를 산출함.

**[A] Kendall–\(\tau_b\) 기반 (4종)**  
- $$C_{\tau}^{(m),\mathrm{full}}(t)$$  
- $$C_{\tau}^{(m),\mathrm{ties3}}(t)$$  
- $$C_{\tau}^{(m),\mathrm{ties5}}(t)$$  
- $$C_{\tau}^{(m),\mathrm{ties10}}(t)$$  

**[B] RBO 기반 (1종)**  
- $$C_{\mathrm{RBO}}^{(m)}(t)$$  

**[C] Mallows 기반 (1종)**  
- $$C_{\mathrm{M}}^{(m)}(t)$$  

본 구성은 각 옵션의 수학적 정의와 해석 목적에 정합적인 입력만을 선택한 결과이며, 모델별 설명 합의 구조를 일관되게 비교할 수 있도록 설계됨.

## <mark> 5) 거래 필터링 검증

###  5-1) 성능 비교를 위한 거래 필터링 평가 설계
* 모델별로 모든 C-index 변형에 대해 동일한 거래 필터링 정책을 적용한다.
* 모델이 거래 신호를 산출하더라도, 해당 시점의 **모델별 C-index**가 임계값 $\theta^{(m)}$ 미만인 경우 거래를 제외한다.
* 자산 및 모델별 C-index 분포 차이를 제거하기 위해, 임계값은 고정값이 아닌 **모델별 C-index 분포의 q-quantile 기준**으로 설정한다.
* 필터링은 거래 빈도를 줄이는 정책이므로, 성과 지표는 반드시 Trade Count와 함께 해석한다. 특히 강한 필터링은 소표본 편향을 유발할 수 있으므로 pooled 분석에서는 $N \ge 15$ 조건을 적용한다.

본 절의 목적은 가능한 모든 조합을 기계적으로 비교하는 것이 아니라, **C-index가 낮은 구간을 제거했을 때 위험 대비 성과가 개선되는지**를 확인하는 데 있다. 다만 현재 결과는 모델별로 차이가 있으므로, 이후 해석에서는 일관된 입증보다 "부분적 개선과 한계"를 함께 보고한다.




In [ ]:
from scipy.stats import kendalltau

# 5-1) C-index Calculation Logic
def rbo_score_finite(rank1, rank2, p=0.9):
    K = min(len(rank1), len(rank2))
    seen1, seen2, weighted_sum = set(), set(), 0.0
    for d in range(1, K + 1):
        seen1.add(rank1[d-1]); seen2.add(rank2[d-1])
        weighted_sum += ((1 - p) * (p ** (d - 1)) * (len(seen1 & seen2) / d))
    return weighted_sum / (1 - (p ** K))

def safe_kendall_tau(x, y):
    tau, _ = kendalltau(x, y)
    return max(0.0, float(tau)) if tau and not pd.isna(tau) else 0.0

def mallows_proxy_from_tau(r1, r2):
    tau = safe_kendall_tau(r1, r2)
    return float(np.exp(-(1 - tau) / 2))

def calculate_c_indices(row):
    results = {'C_tau_full': safe_kendall_tau(row['permFullRank'], row['shapFullRank'])}
    for K in [3, 5, 10]:
        results[f'C_tau_ties{K}'] = safe_kendall_tau(row[f'permTies{K}'], row[f'shapTies{K}'])
    results['C_RBO_full'] = rbo_score_finite(row['permFullRank'], row['shapFullRank'])
    results['C_Mallows_full'] = mallows_proxy_from_tau(row['permFullRank'], row['shapFullRank'])
    return pd.Series(results)

# 5-2) Batch Processing of C-index calculation
analysisDf = pd.concat([
    rankingsDf[['model', 'asset', 'date']],
    rankingsDf.apply(calculate_c_indices, axis=1)
], axis=1)
analysisDf['date'] = pd.to_datetime(analysisDf['date'])

### 5-2) 필터링 정책

* 모델이 buy(또는 trade) 신호를 산출하더라도, 해당 시점의 모델별 C-index \(C^{(m)}(t)\)가 임계값 \(\theta^{(m)}\) 미만인 경우 거래를 수행하지 않음
* 임계값 \(\theta^{(m)}\)는 고정값 또는 모델별 C-index 분포의 q-quantile 기준으로 설정하며, 하위 q 구간을 제거하고 상위 \(1-q\) 구간만 거래하도록 설계함


In [ ]:
# 5-2) Model Extension (gbm / rf / mlp) + Multi-quantile Sensitivity Analysis
#      OEP Prediction Signal Generation + C-index Merging + Filtering Functions
#      artifact 저장 위치: /content/drive/MyDrive/c-index/artifacts 바로 아래

from pathlib import Path
import numpy as np
import pandas as pd

ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TRADING_DAYS_PER_YEAR = 252
SHARPE_EPSILON = 1e-6

FINAL_DF_CACHE = ARTIFACT_DIR / 'finalDf_cache.pkl'
OEP_SIGNALS_CACHE = ARTIFACT_DIR / 'oep_signals_cache.pkl'
ANALYSIS_DF_CACHE = ARTIFACT_DIR / 'analysisDf_cache.pkl'
OEP_DF_CACHE = ARTIFACT_DIR / 'oepDf_cache.pkl'

USE_EVAL_CACHE = False  # fresh training/XAI run: do not reuse a previous finalDf
WRITE_EVAL_CACHE = True


def _cache_exists(path):
    return Path(path).exists()


def _read_cache(path):
    return pd.read_pickle(path)


def _write_cache(df, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_pickle(path)
    print(f'[cache saved] {path}')


def build_oep_signals_for_model(model_name, oep_df):
    sig = oep_df[['asset', 'date', 'ret_1d_next']].copy()
    sig['date'] = pd.to_datetime(sig['date'])
    sig['prob'] = predict_proba_model(
        model_name=model_name,
        Xraw=oep_df[feature_universe + ['asset_cat']].copy()
    )
    sig['y_hat'] = (sig['prob'] >= thrMap[model_name]).astype(int)
    sig['model'] = model_name
    return sig[['model', 'asset', 'date', 'ret_1d_next', 'prob', 'y_hat']]


if USE_EVAL_CACHE and _cache_exists(FINAL_DF_CACHE):
    finalDf = _read_cache(FINAL_DF_CACHE)
    finalDf['date'] = pd.to_datetime(finalDf['date'])
    finalDf = finalDf.sort_values(['model', 'asset', 'date']).reset_index(drop=True)
    if _cache_exists(OEP_SIGNALS_CACHE):
        oep_signals = _read_cache(OEP_SIGNALS_CACHE)
        oep_signals['date'] = pd.to_datetime(oep_signals['date'])
    if _cache_exists(ANALYSIS_DF_CACHE):
        analysisDf = _read_cache(ANALYSIS_DF_CACHE)
        analysisDf['date'] = pd.to_datetime(analysisDf['date'])
    if _cache_exists(OEP_DF_CACHE):
        oepDf = _read_cache(OEP_DF_CACHE)
        oepDf['date'] = pd.to_datetime(oepDf['date'])
    print(f'[cache loaded] finalDf: {FINAL_DF_CACHE} | shape={finalDf.shape}')
else:
    oep_signals = pd.concat([
        build_oep_signals_for_model('gbm', oepDf),
        build_oep_signals_for_model('rf',  oepDf),
        build_oep_signals_for_model('mlp', oepDf)
    ], axis=0).reset_index(drop=True)

    finalDf = pd.merge(analysisDf, oep_signals, on=['model', 'asset', 'date'], how='inner')
    finalDf['date'] = pd.to_datetime(finalDf['date'])
    finalDf = finalDf.sort_values(['model', 'asset', 'date']).reset_index(drop=True)

if WRITE_EVAL_CACHE:
    _write_cache(finalDf, FINAL_DF_CACHE)
    if 'oep_signals' in globals():
        _write_cache(oep_signals, OEP_SIGNALS_CACHE)
    if 'analysisDf' in globals():
        _write_cache(analysisDf, ANALYSIS_DF_CACHE)
    if 'oepDf' in globals():
        _write_cache(oepDf, OEP_DF_CACHE)
    for _name in ['panelDf', 'labeledDf']:
        if _name in globals():
            _write_cache(globals()[_name], ARTIFACT_DIR / f'{_name}_cache.pkl')

print('[artifact path]')
print('ARTIFACT_DIR =', ARTIFACT_DIR)
print('artifact files:', sorted(p.name for p in ARTIFACT_DIR.iterdir() if p.is_file()))


# N < 15 시 Sharpe NaN 처리 — 소표본 편향(분모 0 수렴 -> Sharpe 폭발) 방지
# q=0.4~0.5 강한 필터링시 거래 2~5건으로 급감하는 현상을 명시적으로 차단함
MIN_TRADES_POOLED = 15   # Pooled: 통계적 유의성 기준
MIN_TRADES_ASSET  = 5    # Asset-wise: 본질적 소표본이므로 완화, Win Rate 중심 해석


def _prepare_trade_frame(trades, return_col='ret_1d_next', date_col='date'):
    """Return a normalized trade DataFrame with ret/date columns when available."""
    if isinstance(trades, pd.DataFrame):
        out = trades.copy()
        if return_col not in out.columns:
            raise KeyError(f"'{return_col}' column is required for metric calculation.")
        out = out.rename(columns={return_col: 'ret'})
        if date_col in out.columns:
            out['date'] = pd.to_datetime(out[date_col])
        else:
            out['date'] = pd.NaT
        return out[['date', 'ret']].dropna(subset=['ret'])

    out = pd.DataFrame({'ret': pd.Series(trades).dropna()})
    out['date'] = pd.NaT
    return out[['date', 'ret']]


def _max_drawdown(return_series):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return np.nan
    equity = returns.cumsum()
    return float((equity - equity.cummax()).min() * 100)


def compute_metrics(trades, min_trades=MIN_TRADES_POOLED, calendar_dates=None):
    """
    Report only two Sharpe variants:
    - Sharpe: per-trade mean/std, not annualized (main metric)
    - Sharpe_annualized: per-trade Sharpe * sqrt(observed annual trade count)

    The old per-trade * sqrt(252) annualization is intentionally not used because
    these are sparse event-driven trades, not daily returns traded every day.
    """
    trade_df = _prepare_trade_frame(trades)
    if trade_df['date'].notna().any():
        trade_df = trade_df.sort_values('date').reset_index(drop=True)

    rets = trade_df['ret'].dropna()
    n = len(rets)

    if calendar_dates is not None:
        eval_days = len(pd.to_datetime(pd.Series(calendar_dates).dropna().unique()))
    elif trade_df['date'].notna().any():
        eval_days = len(pd.to_datetime(trade_df['date'].dropna().unique()))
    else:
        eval_days = np.nan

    if pd.notna(eval_days) and eval_days > 0:
        annual_trades = n / (eval_days / TRADING_DAYS_PER_YEAR)
    else:
        annual_trades = np.nan

    if n == 0:
        return {
            'Trade Count': 0,
            'Avg Return': np.nan,
            'Win Rate': np.nan,
            'Sharpe': np.nan,
            'Sharpe_annualized': np.nan,
            'Annual Trades': float(annual_trades) if pd.notna(annual_trades) else np.nan,
            'Eval Days': int(eval_days) if pd.notna(eval_days) else np.nan,
            'Max Drawdown': np.nan,
            'Note': ''
        }

    avg_ret = rets.mean() * 100
    win_rate = (rets > 0).mean() * 100
    mdd_trade = _max_drawdown(rets)

    if n < min_trades:
        sharpe_pt = np.nan
        sharpe_ann = np.nan
        note = f'N={n}<{min_trades} (소표본 경고: Sharpe 신뢰 불가)'
    else:
        trade_std = rets.std()
        if trade_std <= SHARPE_EPSILON:
            sharpe_pt = np.nan
            sharpe_ann = np.nan
            note = '표준편차≈0 (Sharpe 신뢰 불가)'
        else:
            sharpe_pt = float(rets.mean() / trade_std)
            sharpe_ann = float(sharpe_pt * np.sqrt(annual_trades)) if pd.notna(annual_trades) else np.nan
            note = ''

    return {
        'Trade Count': int(n),
        'Avg Return': float(avg_ret),
        'Win Rate': float(win_rate),
        'Sharpe': sharpe_pt,
        'Sharpe_annualized': sharpe_ann,
        'Annual Trades': float(annual_trades) if pd.notna(annual_trades) else np.nan,
        'Eval Days': int(eval_days) if pd.notna(eval_days) else np.nan,
        'Max Drawdown': float(mdd_trade),
        'Note': note
    }


def evaluate_no_filter(df):
    df = df.copy().sort_values('date')
    trades = df.loc[df['y_hat'] == 1, ['date', 'ret_1d_next']].copy()
    metrics = compute_metrics(trades, min_trades=MIN_TRADES_POOLED, calendar_dates=df['date'].unique())
    metrics['C-Index Type'] = 'No Filter'
    metrics['Quantile'] = np.nan
    return metrics, trades.sort_values('date')['ret_1d_next'].cumsum()


def evaluate_filter(df, c_col, quantile=0.3, min_trades=MIN_TRADES_POOLED):
    df = df.copy().sort_values('date')
    model_thr = df.groupby('model')[c_col].quantile(quantile).to_dict()
    df['thr'] = df['model'].map(model_thr)
    df['is_trade'] = (df['y_hat'] == 1) & (df[c_col] >= df['thr'])
    trades = df.loc[df['is_trade'], ['date', 'ret_1d_next']].copy()
    metrics = compute_metrics(trades, min_trades=min_trades, calendar_dates=df['date'].unique())
    metrics['C-Index Type'] = c_col
    metrics['Quantile'] = quantile
    return metrics, trades.sort_values('date')['ret_1d_next'].cumsum()







### 5-3) 성과평가

* **비교군**
  * No filter: 모델 신호 전부 거래
  * C-index filter: 모델별 C-index 기준 quantile 필터링 적용 (q ∈ {0.2, 0.3, 0.4, 0.5})

* **평가 지표**
  * `Sharpe`: per-trade Sharpe (주 지표, annualize 없음)
  * `Sharpe_annualized`: `per-trade Sharpe × sqrt(연간 평균 거래 수)`로 계산한 event annualized Sharpe (보조 지표)
  * Avg Return, Win Rate, Max Drawdown, Trade Count

* **통계적 신뢰성 확보**
  * 강한 필터링(q = 0.4~0.5) 적용 시 거래 횟수가 급감하여 수익률의 표준편차(분모)가 0에 수렴하고, Sharpe ratio가 수학적으로 폭발하는 **소표본 편향(small-sample bias)**이 발생할 수 있다.
  * 이를 방지하기 위해 pooled 평가에서는 **최소 거래 횟수 $N \ge 15$ 조건**을 만족하는 시나리오 중에서만 Sharpe 기준 최적 시나리오를 선정하며, 조건 미달 시 Sharpe를 NaN으로 처리한다.
  * Sharpe 분모에는 수치 안정성을 위한 epsilon($10^{-6}$)을 사용하되, 표준편차가 epsilon 이하이면 Sharpe를 NaN으로 처리한다.

* **실행 결과 기준 해석 원칙**
  * main table에서는 `Sharpe`(per-trade)와 `Trade Count`를 함께 해석한다.
  * annualized Sharpe는 `sqrt(252)`가 아니라 관측 기간의 연간 평균 거래 수를 사용해 계산하며, 보조 지표로만 사용한다.
  * STEP=2는 관측 window를 촘촘히 sampling해 추정 안정성을 높이지만, window overlap 때문에 독립 표본 수가 동일 비율로 증가한 것은 아니다.
  * 따라서 현 단계의 해석은 "C-index 필터링이 일부 모델에서 위험 대비 성과 개선 가능성을 보인다"로 제한하며, 통계적 유의성은 block bootstrap, permutation test, multiple testing correction을 통해 추가 검증해야 한다.



In [ ]:
save_path = str(FIGURES_DIR)
os.makedirs(save_path, exist_ok=True)

q_list = [0.2, 0.3, 0.4, 0.5]
results = []

c_index_cols = finalDf.filter(like='C_', axis=1)

# Baseline (pooled)
for model_name, sub in finalDf.groupby('model'):
    sub = sub.copy().sort_values('date')
    base_trades = sub.loc[sub['y_hat'] == 1, ['date', 'ret_1d_next']]
    m = compute_metrics(base_trades, min_trades=MIN_TRADES_POOLED, calendar_dates=sub['date'].unique())
    m.update({'Model': model_name, 'C-Index Type': 'No Filter', 'Quantile': np.nan})
    results.append(m)

# Filtering (pooled)
for model_name, sub in finalDf.groupby('model'):
    sub = sub.copy().sort_values('date')
    for col in c_index_cols.columns:
        for q in q_list:
            m, _ = evaluate_filter(sub, col, quantile=q, min_trades=MIN_TRADES_POOLED)
            m.update({'Model': model_name, 'C-Index Type': col, 'Quantile': q})
            results.append(m)

summaryTable = pd.DataFrame(results)

baseline_trade_counts = (
    summaryTable[summaryTable['C-Index Type'] == 'No Filter']
    .set_index('Model')['Trade Count']
    .to_dict()
)
summaryTable['Baseline Trade Count'] = summaryTable['Model'].map(baseline_trade_counts)
summaryTable['Delta Trade Count'] = summaryTable['Trade Count'] - summaryTable['Baseline Trade Count']

# Active filter only: N >= 15, finite Sharpe, and an actual reduction in trades.
# This excludes no-op C-index candidates whose filtered trade count equals baseline.
valid_pool = summaryTable[
    (summaryTable['C-Index Type'] != 'No Filter') &
    (summaryTable['Trade Count'] >= MIN_TRADES_POOLED) &
    (summaryTable['Delta Trade Count'] < 0) &
    np.isfinite(summaryTable['Sharpe'])
]

if valid_pool.empty:
    print("WARNING: active-filter 조건을 만족하는 시나리오가 없습니다.")
    best_rows = pd.DataFrame()
else:
    best_rows = (
        valid_pool
        .sort_values(['Model', 'Sharpe'], ascending=[True, False])
        .groupby('Model')
        .first()
    )

display(summaryTable)
display(best_rows)




In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

save_path = str(FIGURES_DIR)
os.makedirs(save_path, exist_ok=True)

def save_df_pretty(df, filename, dpi=450, title=None):
    df_show = df.copy()

    if isinstance(df_show, pd.Series):
        df_show = df_show.to_frame()

    index_name = df_show.index.name if df_show.index.name else "Model"
    df_show = df_show.reset_index().rename(columns={df_show.columns[0]: index_name})

    nrows, ncols = df_show.shape
    fig_w = max(10, ncols * 2.3)
    fig_h = max(2.5, (nrows + 1) * 0.6)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")

    if title:
        ax.set_title(title, fontsize=13, pad=14)

    tbl = ax.table(
        cellText=df_show.values,
        colLabels=df_show.columns,
        loc="center",
        cellLoc="center"
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.1, 1.6)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(1.2)
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_height(cell.get_height() * 1.15)
        if c == 0:
            cell.set_text_props(weight="bold")

    plt.tight_layout(pad=1.5)
    fig.savefig(os.path.join(save_path, filename), dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)


save_df_pretty(summaryTable, "summary_performance_table.png", dpi=450,
               title="Summary Performance Table")

save_df_pretty(best_rows, "best_scenarios_by_model.png", dpi=450,
               title="Best Scenarios by Model")

print("saved clean tables (450 dpi)")

### 5-4) 자산별 결과

자산별 분석에서는 개별 자산의 표본 크기가 본질적으로 작아 Sharpe 수치의 통계적 신뢰도가 낮으므로, **승률(Win Rate) 개선 폭**과 **평균 수익률(Avg Return)**을 중심 지표로 활용한다.

실행 결과상 자산별 best scenario에서는 다수의 자산에서 Win Rate 개선이 관찰된다. GBM은 3개 자산 모두에서 Win Rate가 개선되거나 유지되었고, RF도 3개 자산 모두에서 Win Rate가 개선되었다. MLP는 SPY와 TLT에서 Win Rate가 개선되었으나 GLD는 baseline과 동일했다. MDD 역시 RF의 3개 자산에서 모두 개선되는 등 긍정적 방향성이 일부 확인된다.

그러나 자산별 결과는 표본 수가 매우 작다. 예를 들어 GBM GLD는 거래 1건, MLP TLT는 거래 2건, GBM TLT는 거래 3건에 불과하다. 이 경우 승률 100% 또는 MDD 0은 전략의 안정적 우수성이라기보다 소표본에서 쉽게 발생하는 값일 수 있다. 따라서 자산별 결과는 **범용성의 확정적 증거가 아니라, 추가 표본 확대 후 재검증해야 할 방향성**으로 해석한다.

결론적으로 자산별 분석은 C-index 필터링이 특정 자산에서만 완전히 무력한 것은 아니라는 가능성을 보여주지만, 현재 표본만으로 "범용적 리스크 관리 도구임을 증명한다"고 단정하기에는 부족하다. 이후 STEP 축소 또는 OEP 확장을 통해 자산별 거래 수를 늘린 뒤 동일한 개선 패턴이 유지되는지 확인해야 한다.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Asset-wise Results
# 자산별: 표본이 본질적으로 적어 Sharpe 신뢰 어려움 -> Win Rate + Avg Return 중심

asset_results = []

for (model_name, asset_name), sub in finalDf.groupby(['model', 'asset']):
    sub = sub.copy().sort_values('date').reset_index(drop=True)
    base_trades = sub.loc[sub['y_hat'] == 1, ['date', 'ret_1d_next']].copy()
    base_metrics = compute_metrics(base_trades, min_trades=MIN_TRADES_ASSET, calendar_dates=sub['date'].unique())
    base_metrics['Model'] = model_name
    base_metrics['Asset'] = asset_name
    base_metrics['C-Index Type'] = 'No Filter'
    base_metrics['Quantile'] = np.nan
    asset_results.append(base_metrics)

for (model_name, asset_name), sub in finalDf.groupby(['model', 'asset']):
    sub = sub.copy().sort_values('date').reset_index(drop=True)
    for col in c_index_cols.columns:
        for q in q_list:
            thr = sub[col].quantile(q)
            tmp = sub.copy()
            tmp['is_trade'] = (tmp['y_hat'] == 1) & (tmp[col] >= thr)
            trades = tmp.loc[tmp['is_trade'], ['date', 'ret_1d_next']].copy()
            m = compute_metrics(trades, min_trades=MIN_TRADES_ASSET, calendar_dates=sub['date'].unique())
            m['Model'] = model_name
            m['Asset'] = asset_name
            m['C-Index Type'] = col
            m['Quantile'] = q
            m['Threshold Value'] = thr
            asset_results.append(m)

assetSummaryTable = pd.DataFrame(asset_results)
assetSummaryTable = assetSummaryTable[
    ['Model', 'Asset', 'C-Index Type', 'Quantile',
     'Trade Count', 'Avg Return', 'Win Rate', 'Sharpe', 'Sharpe_annualized', 'Annual Trades', 'Max Drawdown', 'Note']
].sort_values(['Model', 'Asset', 'C-Index Type', 'Quantile']).reset_index(drop=True)

print("=== Asset-wise Performance Comparison ===")
display(assetSummaryTable)

# Best Scenario: Win Rate 기준 (Sharpe 소표본 편향 회피)
# 자산별 분석에서는 Sharpe 대신 Win Rate의 일관된 개선으로 범용성 확인
bestAssetRows = (
    assetSummaryTable[assetSummaryTable['C-Index Type'] != 'No Filter']
    .sort_values(['Model', 'Asset', 'Win Rate'], ascending=[True, True, False])
    .groupby(['Model', 'Asset'])
    .head(1)
    .reset_index(drop=True)
)

print("=== Best Scenario by Win Rate (per Model x Asset) ===")
display(bestAssetRows)

# Improvement over No Filter
baselineAssetRows = (
    assetSummaryTable[assetSummaryTable['C-Index Type'] == 'No Filter']
    [['Model', 'Asset', 'Win Rate', 'Avg Return', 'Sharpe', 'Sharpe_annualized', 'Max Drawdown', 'Trade Count']]
    .rename(columns={
        'Win Rate': 'Baseline Win Rate',
        'Avg Return': 'Baseline Avg Return',
        'Sharpe': 'Baseline Sharpe',
        'Sharpe_annualized': 'Baseline Sharpe Annualized',
        'Max Drawdown': 'Baseline Max Drawdown',
        'Trade Count': 'Baseline Trade Count'
    })
)

assetImprovementTable = pd.merge(bestAssetRows, baselineAssetRows, on=['Model', 'Asset'], how='left')
assetImprovementTable['Delta Win Rate']    = assetImprovementTable['Win Rate']     - assetImprovementTable['Baseline Win Rate']
assetImprovementTable['Delta Avg Return']  = assetImprovementTable['Avg Return']   - assetImprovementTable['Baseline Avg Return']
assetImprovementTable['Delta Sharpe']      = assetImprovementTable['Sharpe']       - assetImprovementTable['Baseline Sharpe']
assetImprovementTable['Delta MDD']         = assetImprovementTable['Max Drawdown'] - assetImprovementTable['Baseline Max Drawdown']
assetImprovementTable['Delta Trade Count'] = assetImprovementTable['Trade Count']  - assetImprovementTable['Baseline Trade Count']

assetImprovementTable = assetImprovementTable[[
    'Model', 'Asset', 'C-Index Type', 'Quantile',
    'Baseline Win Rate', 'Win Rate', 'Delta Win Rate',
    'Baseline Avg Return', 'Avg Return', 'Delta Avg Return',
    'Baseline Sharpe', 'Sharpe', 'Delta Sharpe',
    'Baseline Max Drawdown', 'Max Drawdown', 'Delta MDD',
    'Baseline Trade Count', 'Trade Count', 'Delta Trade Count',
    'Note'
]].sort_values(['Model', 'Asset']).reset_index(drop=True)

print("=== Improvement over No Filter (per Model x Asset) ===")
display(assetImprovementTable.round(4))

# Robustness Count — Win Rate & Avg Return 기준
robustnessSummary = (
    assetImprovementTable
    .assign(
        WinRate_Improved=lambda x: x['Delta Win Rate'] > 0,
        AvgReturn_Improved=lambda x: x['Delta Avg Return'] > 0,
        MDD_Improved=lambda x: x['Delta MDD'] > 0
    )
    .groupby('Model')[['WinRate_Improved', 'AvgReturn_Improved', 'MDD_Improved']]
    .sum()
    .reset_index()
)
robustnessSummary['MDD_Equal(Delta=0)'] = (
    assetImprovementTable.assign(eq=lambda x: x['Delta MDD'] == 0)
    .groupby('Model')['eq'].sum().values
)
robustnessSummary['MDD_Worsened(Delta<0)'] = (
    assetImprovementTable.assign(wo=lambda x: x['Delta MDD'] < 0)
    .groupby('Model')['wo'].sum().values
)

print("=== Robustness Count by Model (Win Rate & Avg Return 기준) ===")
display(robustnessSummary)

# 시각화: Win Rate + Avg Return 비교 (Sharpe 대신)
for model_name in sorted(assetImprovementTable['Model'].unique()):
    sub = assetImprovementTable[assetImprovementTable['Model'] == model_name].copy()
    x = np.arange(len(sub))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(x - width/2, sub['Baseline Win Rate'], width, label='No Filter', color='steelblue')
    axes[0].bar(x + width/2, sub['Win Rate'],          width, label='Best Filtered', color='darkorange')
    axes[0].set_xticks(x); axes[0].set_xticklabels(sub['Asset'].tolist())
    axes[0].set_ylabel('Win Rate (%)'); axes[0].legend(); axes[0].grid(axis='y')
    axes[0].set_title(f'{model_name}: Asset-wise Win Rate')

    axes[1].bar(x - width/2, sub['Baseline Avg Return'], width, label='No Filter', color='steelblue')
    axes[1].bar(x + width/2, sub['Avg Return'],           width, label='Best Filtered', color='darkorange')
    axes[1].set_xticks(x); axes[1].set_xticklabels(sub['Asset'].tolist())
    axes[1].set_ylabel('Avg Return (%)'); axes[1].legend(); axes[1].grid(axis='y')
    axes[1].set_title(f'{model_name}: Asset-wise Avg Return')

    plt.suptitle(f'[{model_name}] Win Rate & Avg Return (Sharpe는 소표본 편향으로 참고용 전환)', fontsize=9, color='gray')
    plt.tight_layout()
    plt.show()





In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

save_path = str(FIGURES_DIR)
os.makedirs(save_path, exist_ok=True)

def save_df_pretty(df, filename, dpi=450, title=None, include_index=True, index_name="Model"):
    df_show = df.copy()
    if isinstance(df_show, pd.Series):
        df_show = df_show.to_frame()
    if include_index:
        actual_index_name = df_show.index.name if df_show.index.name else index_name
        df_show = df_show.reset_index().rename(columns={df_show.columns[0]: actual_index_name})
    nrows, ncols = df_show.shape
    fig_w = max(10, ncols * 1.8)
    fig_h = max(2.5, (nrows + 1) * 0.52)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=13, pad=14)
    tbl = ax.table(cellText=df_show.values, colLabels=df_show.columns, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.08, 1.55)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(1.1)
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_height(cell.get_height() * 1.12)
        if include_index and c == 0:
            cell.set_text_props(weight="bold")
    plt.tight_layout(pad=1.5)
    fig.savefig(os.path.join(save_path, filename), dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)

save_df_pretty(summaryTable, "summary_performance_table.png", dpi=450,
               title="Summary Performance Table (N<15 -> Sharpe=NaN)", include_index=False)
save_df_pretty(best_rows, "best_scenarios_by_model.png", dpi=450,
               title="Best Active Scenarios by Model (N >= 15, Trade Count Reduced)", include_index=True)
save_df_pretty(assetSummaryTable, "asset_summary_performance_table.png", dpi=450,
               title="Asset-wise Performance Comparison", include_index=False)
save_df_pretty(bestAssetRows, "best_asset_scenarios_table.png", dpi=450,
               title="Best Scenario by Win Rate (per Model x Asset)", include_index=False)
save_df_pretty(assetImprovementTable.round(4), "asset_improvement_table.png", dpi=450,
               title="Improvement over No Filter — Win Rate & Avg Return 중심", include_index=False)
save_df_pretty(robustnessSummary, "robustness_summary_table.png", dpi=450,
               title="Robustness Count by Model (Win Rate & Avg Return 기준)", include_index=False)

for model_name in sorted(assetImprovementTable['Model'].unique()):
    sub = assetImprovementTable[assetImprovementTable['Model'] == model_name].copy()
    x = np.arange(len(sub))
    width = 0.35
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(x - width/2, sub['Baseline Win Rate'], width, label='No Filter', color='steelblue')
    axes[0].bar(x + width/2, sub['Win Rate'],          width, label='Best Filtered', color='darkorange')
    axes[0].set_xticks(x); axes[0].set_xticklabels(sub['Asset'].tolist())
    axes[0].set_ylabel('Win Rate (%)'); axes[0].legend(); axes[0].grid(axis='y')
    axes[0].set_title(f'{model_name}: Asset-wise Win Rate')
    axes[1].bar(x - width/2, sub['Baseline Avg Return'], width, label='No Filter', color='steelblue')
    axes[1].bar(x + width/2, sub['Avg Return'],           width, label='Best Filtered', color='darkorange')
    axes[1].set_xticks(x); axes[1].set_xticklabels(sub['Asset'].tolist())
    axes[1].set_ylabel('Avg Return (%)'); axes[1].legend(); axes[1].grid(axis='y')
    axes[1].set_title(f'{model_name}: Asset-wise Avg Return')
    plt.suptitle(f'[{model_name}] Win Rate & Avg Return (Sharpe -> 소표본 편향으로 참고용)', fontsize=9, color='gray')
    plt.tight_layout()
    fig.savefig(os.path.join(save_path, f"model_{model_name}_asset_wise_winrate_comparison.png"),
                dpi=450, bbox_inches='tight', facecolor='white')
    plt.close(fig)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
table_outputs = {
    'overlap_summary.csv': (overlapSummary, True),
    'disagreement_summary.csv': (disagreeSummary, True),
    'asset_disagreement_summary.csv': (assetDisagreeSummary, False),
    'summary_performance.csv': (summaryTable, False),
    'best_scenarios_by_model.csv': (best_rows, True),
    'asset_summary_performance.csv': (assetSummaryTable, False),
    'best_asset_scenarios.csv': (bestAssetRows, False),
    'asset_improvement.csv': (assetImprovementTable, False),
    'robustness_summary.csv': (robustnessSummary, False),
}
for filename, (table, include_index) in table_outputs.items():
    table.to_csv(TABLES_DIR / filename, index=include_index)

print('saved CSV tables to', TABLES_DIR)
if not DRIVE_AVAILABLE:
    import shutil
    archive_path = shutil.make_archive('/content/c-index_run_outputs', 'zip', root_dir=PROJECT_DIR)
    print('download archive:', archive_path)

print("saved all tables and figures (450 dpi) — Win Rate 기준 재구성 완료")


## <mark> 6) 결과 분석 및 종합 결론 </mark>

본 연구는 모델의 예측 신뢰도를 정량화하는 C-index 필터링의 유효성을 3개 모델(LGBM, RF, MLP)과 3개 자산(SPY, TLT, GLD)에 대해 검토하였다. 이번 실행 결과는 재현성 및 Sharpe 계산 안정성 문제를 상당 부분 해결했다는 점에서는 의미가 있으나, C-index 필터링의 성과 개선 효과는 모델별로 다르게 나타났다.

### 6-1) 재현성 및 계산 신뢰성 확보

* 전역 seed, LightGBM/RF/MLP deterministic 설정, PyTorch CPU 고정, yfinance 데이터 캐시를 적용하여 실행 환경의 비결정성을 줄였다.
* 동일 데이터 캐시 조건에서 run1/run2 두 차례 반복 실행한 결과, validation metrics, MLP epoch log, disagreement table, pooled/asset-wise performance가 동일하게 재현되었다.
* 기존에 관찰되던 비현실적 Sharpe 폭발은 강한 필터링 후 거래 수가 2~5건 수준으로 줄어드는 소표본 편향 때문으로 진단되었으며, pooled 분석에서는 $N \ge 15$ 조건을 적용하여 이를 방지하였다.

### 6-2) 설명 불일치와 C-index 필요성

* Top-3 disagreement는 GBM 0.8621, RF 0.8793, MLP 0.7586으로 높게 나타났다.
* 이는 동일 모델·동일 시점에서도 Permutation Importance와 SHAP이 서로 다른 핵심 피처 구조를 제시한다는 뜻이며, 예측 확률만으로는 모델 판단 근거의 안정성을 확인하기 어렵다는 문제의식을 뒷받침한다.
* 따라서 C-index는 예측력 자체를 높이는 지표라기보다, 예측 신호의 설명 구조가 안정적인지 점검하는 **reliability screening index**로 해석하는 것이 적절하다.

### 6-3) Pooled 성과: 부분적 개선

| Model | No Filter Sharpe | Best Filter Sharpe | No Filter MDD | Best Filter MDD | 해석 |
|---|---:|---:|---:|---:|---|
| GBM | 0.1839 | 0.1839 | -3.1166 | -3.1166 | 실질적 개선 없음 |
| MLP | 0.2297 | 0.3700 | -6.3981 | -3.1597 | 개선 있으나 Trade Count 15로 주의 필요 |
| RF | 0.0501 | 0.1457 | -10.0922 | -4.4708 | 가장 안정적인 개선 패턴 |

* GBM은 필터링 후 best scenario가 baseline과 동일하여 C-index 필터링 효과가 확인되지 않았다.
* MLP는 Avg Return, Win Rate, Sharpe, MDD가 개선되었으나, Trade Count가 25건에서 15건으로 감소하여 결과 안정성에 주의가 필요하다.
* RF는 Trade Count가 37건에서 24건으로 줄었지만 Sharpe와 MDD가 함께 개선되어, 현재 결과 중 가장 설득력 있는 사례다.

### 6-4) 최적 C-index 및 q에 대한 판단

* 이번 실행 결과에서는 하나의 C-index 변형이 모든 모델에서 우세하지 않았다. GBM은 `C_tau_full`, MLP는 `C_tau_ties10`, RF는 `C_tau_full`이 pooled best로 선택되었다.
* 따라서 기존처럼 `Mallows_full`을 최적 변형으로 단정하기 어렵다.
* q 역시 모델별로 GBM 0.2, MLP 0.4, RF 0.5가 선택되어, 현재 결과만으로 q=0.2~0.3을 universal sweet spot으로 주장하기는 어렵다.
* 다만 q가 높아질수록 거래 수가 줄어 소표본 위험이 커진다는 점은 확인되므로, 향후 분석에서는 성과뿐 아니라 Trade Count와 통계적 유의성을 함께 기준으로 삼아야 한다.

### 6-5) 자산별 분석: 방향성은 있으나 표본 부족

* 자산별 best scenario에서는 Win Rate 및 MDD 개선 사례가 다수 관찰된다.
* 그러나 GBM GLD 1건, MLP TLT 2건, GBM TLT 3건처럼 거래 수가 매우 작은 케이스가 포함되어 있어, 자산별 결과를 확정적 증거로 해석하기 어렵다.
* 따라서 자산별 분석은 "특정 자산에서만 효과가 나타난 것은 아니다"라는 참고 근거로는 사용할 수 있으나, 범용성 입증을 위해서는 STEP 축소 또는 OEP 확장을 통한 표본 확대가 필요하다.

### 6-6) 최종 요약

| 항목 | 현재 실행 결과 기준 판단 |
|---|---|
| 재현성 | run1/run2 기준 핵심 출력 동일, 3회 검증은 추후 확장 가능 |
| 설명 불일치 | Top-3 disagreement 0.76~0.88 수준으로 높아 C-index 도입 필요성 지지 |
| Pooled 필터링 효과 | RF와 MLP에서 부분적 개선, GBM은 개선 없음 |
| 최적 C-index 변형 | 모델별로 달라 단일 최적 변형 단정 불가 |
| 최적 q 구간 | 모델별로 달라 universal sweet spot 단정 불가 |
| 자산별 범용성 | 개선 방향성은 있으나 거래 수가 작아 추가 검증 필요 |
| 다음 단계 | STEP=5 또는 STEP=3으로 표본 확대 후 동일 분석 재수행, 이후 bootstrap/permutation test 및 거래비용 반영 |

결론적으로, 이번 결과는 **C-index 기반 거래 필터링이 완전히 입증되었다**기보다는, **설명 불일치가 실제로 높게 존재하며 이를 활용한 필터링이 일부 모델에서 위험 대비 성과 개선 가능성을 보인다**는 수준으로 해석하는 것이 타당하다. 논문에서는 C-index를 alpha 생성 도구가 아니라 예측 신호의 신뢰 가능 구간을 선별하는 reliability screening layer로 위치시키는 것이 가장 안전하다.

